In [5]:

import pandas as pd
from pathlib import Path

# Project folders
DATA_PATH = Path("../data/raw/")
OUTPUT_PATH = Path("../outputs/tables/")

# Create output folder if it does not exist
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Load all six datasets
ball = pd.read_csv(DATA_PATH / "ball_by_ball.csv")
matches = pd.read_csv(DATA_PATH / "matches.csv")
competitions = pd.read_csv(DATA_PATH / "competitions.csv")
teams = pd.read_csv(DATA_PATH / "teams.csv")
playing_xi = pd.read_csv(DATA_PATH / "playing_eleven.csv")
player_mapping = pd.read_csv(
    DATA_PATH / "competition_team_player_mapping.csv"
)

# Create dataset dictionary
datasets = {
    "ball_by_ball": ball,
    "matches": matches,
    "competitions": competitions,
    "teams": teams,
    "playing_eleven": playing_xi,
    "player_mapping": player_mapping
}

print("All six datasets loaded successfully!")

for name, df in datasets.items():
    print(f"{name}: {df.shape}")


All six datasets loaded successfully!
ball_by_ball: (200999, 33)
matches: (883, 22)
competitions: (141, 4)
teams: (353, 3)
playing_eleven: (28381, 9)
player_mapping: (6413, 6)


In [6]:

# Missing-value report

missing_reports = []

for name, df in datasets.items():
    for column in df.columns:
        missing = df[column].isna().sum()

        missing_reports.append({
            "Dataset": name,
            "Column": column,
            "Missing_Count": missing,
            "Missing_Percentage": round(
                (missing / len(df)) * 100, 2
            )
        })

missing_report = pd.DataFrame(missing_reports)

missing_report = missing_report.sort_values(
    ["Dataset", "Missing_Percentage"],
    ascending=[True, False]
)

display(missing_report)

# Save report
missing_report.to_csv(
    OUTPUT_PATH / "missing_value_report.csv",
    index=False
)

print("Missing-value report saved successfully!")


,Dataset,Column,Missing_Count,Missing_Percentage
20,ball_by_ball,dismissal_type_id,190118,94.59
21,ball_by_ball,dismissal_type,190118,94.59
22,ball_by_ball,dismissed_player_id,190107,94.58
23,ball_by_ball,extras_type,189167,94.11
28,ball_by_ball,stroke,6158,3.06
...,...,...,...,...
69,playing_eleven,is_impact,0,0.00
70,playing_eleven,is_impact_off,0,0.00
59,teams,comp_id,0,0.00
60,teams,team_id,0,0.00


Missing-value report saved successfully!


Ball by Ball: (200999, 33)
Matches: (883, 22)
Competitions: (141, 4)
Teams: (353, 3)
Playing XI: (28381, 9)
Player Mapping: (6413, 6)


In [7]:

missing_reports = []

for name, df in datasets.items():
    for column in df.columns:
        missing = df[column].isna().sum()

        missing_reports.append({
            "Dataset": name,
            "Column": column,
            "Missing_Count": missing,
            "Missing_Percentage": round(
                missing / len(df) * 100, 2
            )
        })

missing_report = pd.DataFrame(missing_reports)

missing_report = missing_report.sort_values(
    ["Dataset", "Missing_Percentage"],
    ascending=[True, False]
)

display(missing_report)

missing_report.to_csv(
    OUTPUT_PATH / "missing_value_report.csv",
    index=False
)

print("Report saved successfully!")


,Dataset,Column,Missing_Count,Missing_Percentage
20,ball_by_ball,dismissal_type_id,190118,94.59
21,ball_by_ball,dismissal_type,190118,94.59
22,ball_by_ball,dismissed_player_id,190107,94.58
23,ball_by_ball,extras_type,189167,94.11
28,ball_by_ball,stroke,6158,3.06
...,...,...,...,...
69,playing_eleven,is_impact,0,0.00
70,playing_eleven,is_impact_off,0,0.00
59,teams,comp_id,0,0.00
60,teams,team_id,0,0.00


Report saved successfully!


In [8]:

key_checks = {
    "Matches": (matches, ["match_id"]),
    "Competitions": (competitions, ["comp_id"]),
    "Teams": (teams, ["comp_id", "team_id"]),
    "Ball By Ball": (
        ball,
        ["match_id", "innings", "innings_ball_num"]
    ),
    "Playing XI": (
        playing_xi,
        ["match_id", "team_id", "player_id"]
    ),
    "Player Mapping": (
        player_mapping,
        ["comp_id", "team_id", "player_id"]
    )
}

results = []

for name, (df, keys) in key_checks.items():
    missing_keys = df[keys].isna().any(axis=1).sum()
    duplicate_keys = df.duplicated(subset=keys).sum()

    results.append({
        "Dataset": name,
        "Candidate_Key": ", ".join(keys),
        "Missing_Key_Rows": missing_keys,
        "Duplicate_Key_Rows": duplicate_keys
    })

key_report = pd.DataFrame(results)

display(key_report)

key_report.to_csv(
    OUTPUT_PATH / "key_validation_report.csv",
    index=False
)


,Dataset,Candidate_Key,Missing_Key_Rows,Duplicate_Key_Rows
0,Matches,match_id,0,0
1,Competitions,comp_id,0,0
2,Teams,"comp_id, team_id",0,0
3,Ball By Ball,"match_id, innings, innings_ball_num",0,0
4,Playing XI,"match_id, team_id, player_id",0,40
5,Player Mapping,"comp_id, team_id, player_id",0,0


In [9]:

# Identify all rows involved in duplicate candidate keys
duplicate_mask = playing_xi.duplicated(
    subset=["match_id", "team_id", "player_id"],
    keep=False
)

duplicate_rows = playing_xi[duplicate_mask].copy()

print("Total rows involved in duplicate keys:", len(duplicate_rows))
print(
    "Number of duplicated key combinations:",
    duplicate_rows[
        ["match_id", "team_id", "player_id"]
    ].drop_duplicates().shape[0]
)

# Display duplicate records
display(
    duplicate_rows.sort_values(
        ["match_id", "team_id", "player_id"]
    )
)

# Check whether the complete rows are identical
print(
    "Exact duplicate rows:",
    playing_xi.duplicated().sum()
)

# Save for investigation
duplicate_rows.to_csv(
    OUTPUT_PATH / "playing_xi_duplicate_records.csv",
    index=False
)


Total rows involved in duplicate keys: 75
Number of duplicated key combinations: 35


,match_id,team_id,player_id,batting_order,is_played,is_captain,is_wk,is_impact,is_impact_off
206,13613,20,223,5.0,True,False,False,0,0
208,13613,20,223,6.0,True,False,False,0,0
200,13613,20,250,2.0,True,True,False,0,0
205,13613,20,250,4.0,True,True,False,0,0
199,13613,20,838,1.0,True,False,False,0,0
...,...,...,...,...,...,...,...,...,...
7344,28962,78,6448,5.0,True,True,False,0,0
7332,28962,78,6674,1.0,True,False,True,0,0
7336,28962,78,6674,2.0,True,False,True,0,0
7340,28962,78,6676,4.0,True,False,False,0,0


Exact duplicate rows: 0


In [10]:

duplicate_groups = (
    duplicate_rows
    .groupby(["match_id", "team_id", "player_id"])
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

display(duplicate_groups)

print("\nColumns in Playing XI:")
print(playing_xi.columns.tolist())


,match_id,team_id,player_id,record_count
30,28962,78,2214,3
29,28962,78,1107,3
28,28962,18,6819,3
26,28962,18,4708,3
31,28962,78,5808,3
0,13613,20,223,2
1,13613,20,250,2
2,13613,20,838,2
3,13613,20,2639,2
5,13613,20,4736,2



Columns in Playing XI:
['match_id', 'team_id', 'player_id', 'batting_order', 'is_played', 'is_captain', 'is_wk', 'is_impact', 'is_impact_off']


In [11]:

# 1. Count exact duplicate rows
print(
    "Exact duplicate rows:",
    playing_xi.duplicated().sum()
)

# 2. Inspect affected matches
affected_match_ids = duplicate_rows["match_id"].unique()

print("\nAffected match IDs:", affected_match_ids)

# 3. Display complete records for affected players
columns_to_check = [
    "match_id",
    "team_id",
    "player_id",
    "batting_order",
    "is_played",
    "is_captain",
    "is_wk",
    "is_impact",
    "is_impact_off"
]

display(
    playing_xi[
        playing_xi["match_id"].isin(affected_match_ids)
    ][columns_to_check].sort_values(
        ["match_id", "team_id", "player_id", "batting_order"]
    ).to_string(index=False)
)

# 4. Inspect match metadata
print("\nMatch dataset columns:")
print(matches.columns.tolist())

display(
    matches[matches["match_id"].isin(affected_match_ids)]
)


Exact duplicate rows: 0

Affected match IDs: [13613 21176 28962]


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off\n    13613       20        223            5.0       True      False False          0              0\n    13613       20        223            6.0       True      False False          0              0\n    13613       20        250            2.0       True       True False          0              0\n    13613       20        250            4.0       True       True False          0              0\n    13613       20        286            9.0       True      False False          0              0\n    13613       20        796            7.0       True      False  True          0              0\n    13613       20        830           11.0       True      False False          0              0\n    13613       20        838            1.0       True      False False          0              0\n    13613       20        838            4.0       True      False False          0              0\


Match dataset columns:
['match_id', 'comp_id', 'match_number', 'match_date', 'stage', 'venue_id', 'venue_country', 'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_team_captain_id', 'away_team_captain_id', 'toss_win_team_id', 'toss_opted', 'man_of_the_match', 'first_inn_revised_overs', 'second_inn_revised_overs', 'revised_target', 'knock_out', 'win_team_id', 'result']


,match_id,comp_id,match_number,match_date,stage,venue_id,venue_country,home_team_id,home_team,away_team_id,...,away_team_captain_id,toss_win_team_id,toss_opted,man_of_the_match,first_inn_revised_overs,second_inn_revised_overs,revised_target,knock_out,win_team_id,result
9,13613,1345,2,2022-02-13,League,44,Australia,20,Australia,27,...,2436.0,27.0,Fielding,830,NaN,NaN,NaN,0,20.0,Australia Won the one over Eliminator
266,21176,1719,1,2023-04-02,League,94,New Zealand,17,New Zealand,27,...,2436.0,17.0,Fielding,0,NaN,NaN,NaN,0,27.0,Sri Lanka won the super over
319,28962,2059,3,2024-01-17,Others,2,India,18,India,78,...,6448.0,18.0,Batting,126,NaN,NaN,NaN,0,18.0,India Won Super Over By 10 Runs


In [12]:

for match_id in affected_match_ids:

    print("\n" + "=" * 60)
    print("MATCH ID:", match_id)
    print("=" * 60)

    match_players = playing_xi[
        playing_xi["match_id"] == match_id
    ].sort_values(
        ["team_id", "batting_order", "player_id"]
    )

    display(match_players)



MATCH ID: 13613


,match_id,team_id,player_id,batting_order,is_played,is_captain,is_wk,is_impact,is_impact_off
199,13613,20,838,1.0,True,False,False,0,0
198,13613,20,3539,1.0,True,False,False,0,0
200,13613,20,250,2.0,True,True,False,0,0
201,13613,20,2639,2.0,True,False,False,0,0
203,13613,20,3539,3.0,True,False,False,0,0
202,13613,20,4736,3.0,True,False,False,0,0
205,13613,20,250,4.0,True,True,False,0,0
204,13613,20,838,4.0,True,False,False,0,0
206,13613,20,223,5.0,True,False,False,0,0
207,13613,20,4736,5.0,True,False,False,0,0



MATCH ID: 21176


,match_id,team_id,player_id,batting_order,is_played,is_captain,is_wk,is_impact,is_impact_off
6106,21176,17,4106,1.0,True,False,False,0,0
6107,21176,17,25534,1.0,True,False,False,0,0
6108,21176,17,3323,2.0,True,False,False,0,0
6109,21176,17,3700,2.0,True,False,False,0,0
6110,21176,17,2246,3.0,True,True,True,0,0
6111,21176,17,3374,3.0,True,False,False,0,0
6112,21176,17,4106,4.0,True,False,False,0,0
6113,21176,17,25534,4.0,True,False,False,0,0
6115,21176,17,3374,5.0,True,False,False,0,0
6114,21176,17,3700,5.0,True,False,False,0,0



MATCH ID: 28962


,match_id,team_id,player_id,batting_order,is_played,is_captain,is_wk,is_impact,is_impact_off
7313,28962,18,126,1.0,True,True,False,0,0
7314,28962,18,6819,1.0,True,False,False,0,0
7317,28962,18,126,2.0,True,True,False,0,0
7316,28962,18,4708,2.0,True,False,False,0,0
7315,28962,18,6819,2.0,True,False,False,0,0
7319,28962,18,47,3.0,True,False,False,0,0
7318,28962,18,2258,3.0,True,False,True,0,0
7320,28962,18,4708,3.0,True,False,False,0,0
7322,28962,18,4755,4.0,True,False,False,0,0
7321,28962,18,6819,4.0,True,False,False,0,0


In [13]:

# Identify IPL competitions
is_ipl = competitions["comp_name"].str.fullmatch(
    r"IPL\s+\d{4}",
    case=False,
    na=False
)

# Identify T20I competitions
is_t20i = competitions["match_format"].eq("T20I")

# Select relevant competitions
selected_competitions = competitions[
    (is_ipl | is_t20i)
].copy()

# Display selected competitions
display(
    selected_competitions.sort_values(
        ["match_format", "season"]
    )
)

print("\nSelected competition count:",
      len(selected_competitions))

print("\nCompetition formats:")
print(selected_competitions["match_format"].value_counts())


,comp_id,comp_name,match_format,season
12,1358,IPL 2022,T20,2022
13,1715,IPL 2023,T20,2023
15,2193,IPL 2024,T20,2024
16,2725,IPL 2025,T20,2025
18,3298,IPL 2026,T20,2026
...,...,...,...,...
136,3128,Sri Lanka in England T20I Series 2026,T20I,2026
137,3446,India in Ireland T20I Series 2026,T20I,2026
138,2906,West Indies in South Africa T20I Series 2026,T20I,2026
139,3334,England in Sri Lanka T20I Series 2026,T20I,2026



Selected competition count: 139

Competition formats:
match_format
T20I    134
T20       5
Name: count, dtype: int64


In [14]:

# Extract starting year from the season column
selected_competitions["season_year"] = pd.to_numeric(
    selected_competitions["season"]
    .astype(str)
    .str.extract(r"(\d{4})")[0],
    errors="coerce"
)

# Keep IPL and T20I competitions from 2022 onwards
selected_competitions = selected_competitions[
    (
        selected_competitions["comp_name"].str.fullmatch(
            r"IPL\s+\d{4}",
            case=False,
            na=False
        )
    ) |
    (
        selected_competitions["match_format"].eq("T20I")
    )
]

selected_competitions = selected_competitions[
    selected_competitions["season_year"] >= 2022
].copy()

# Validation
print("Selected competitions:", len(selected_competitions))

print("\nCompetitions by format:")
print(selected_competitions["match_format"].value_counts())

print("\nCompetitions by year:")
print(
    selected_competitions.groupby(
        ["season_year", "match_format"]
    ).size().to_string()
)

print("\nIPL competitions:")
display(
    selected_competitions[
        selected_competitions["comp_name"].str.startswith(
            "IPL", na=False
        )
    ].sort_values("season_year")
)


Selected competitions: 139

Competitions by format:
match_format
T20I    134
T20       5
Name: count, dtype: int64

Competitions by year:
season_year  match_format
2022         T20              1
             T20I            27
2023         T20              1
             T20I            20
2024         T20              1
             T20I            32
2025         T20              1
             T20I            31
2026         T20              1
             T20I            24

IPL competitions:


,comp_id,comp_name,match_format,season,season_year
12,1358,IPL 2022,T20,2022,2022
13,1715,IPL 2023,T20,2023,2023
15,2193,IPL 2024,T20,2024,2024
16,2725,IPL 2025,T20,2025,2025
18,3298,IPL 2026,T20,2026,2026


In [15]:

# Get selected competition IDs
selected_comp_ids = selected_competitions["comp_id"].unique()

# Filter matches
selected_matches = matches[
    matches["comp_id"].isin(selected_comp_ids)
].copy()

# Filter ball-by-ball records
selected_match_ids = selected_matches["match_id"].unique()

selected_ball = ball[
    ball["match_id"].isin(selected_match_ids)
].copy()

# Save processed datasets
PROCESSED_PATH = Path("../data/processed/")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

selected_competitions.to_csv(
    PROCESSED_PATH / "selected_competitions.csv",
    index=False
)

selected_matches.to_csv(
    PROCESSED_PATH / "selected_matches.csv",
    index=False
)

selected_ball.to_csv(
    PROCESSED_PATH / "selected_ball_by_ball.csv",
    index=False
)

# Final summary
print("FINAL DATASET SUMMARY")
print("---------------------")
print("Competitions:", selected_competitions.shape)
print("Matches:", selected_matches.shape)
print("Ball-by-ball deliveries:", selected_ball.shape)

print("\nMatch years:")
print(
    pd.to_datetime(
        selected_matches["match_date"],
        errors="coerce"
    ).dt.year.value_counts().sort_index()
)


FINAL DATASET SUMMARY
---------------------
Competitions: (139, 5)
Matches: (853, 22)
Ball-by-ball deliveries: (194014, 33)

Match years:
match_date
2022    180
2023    140
2024    185
2025    202
2026    146
Name: count, dtype: int64


In [16]:

# Match IDs present in ball-by-ball data
ball_match_ids = set(selected_ball["match_id"].unique())
match_ids = set(selected_matches["match_id"].unique())

# Matches without ball-by-ball records
matches_without_ball = selected_matches[
    ~selected_matches["match_id"].isin(ball_match_ids)
]

# Deliveries without a matching selected match
orphan_deliveries = selected_ball[
    ~selected_ball["match_id"].isin(match_ids)
]

print("MATCH COVERAGE AUDIT")
print("--------------------")
print("Selected matches:", len(selected_matches))
print("Matches with ball data:", len(ball_match_ids))
print("Matches without ball data:", len(matches_without_ball))
print("Orphan delivery rows:", len(orphan_deliveries))

display(
    matches_without_ball[
        ["match_id", "comp_id", "match_date", "home_team", "away_team"]
    ]
)


MATCH COVERAGE AUDIT
--------------------
Selected matches: 853
Matches with ball data: 837
Matches without ball data: 16
Orphan delivery rows: 0


,match_id,comp_id,match_date,home_team,away_team
396,31281,2193,2024-05-13,Gujarat Titans,Kolkata Knight Riders
399,31284,2193,2024-05-16,Sunrisers Hyderabad,Gujarat Titans
403,31288,2193,2024-05-19,Rajasthan Royals,Kolkata Knight Riders
425,32610,2354,2024-05-22,England,Pakistan
427,32612,2354,2024-05-28,England,Pakistan
447,32622,2398,2024-05-20,Ireland,Scotland
453,38797,2442,2024-12-14,South Africa,Pakistan
481,36564,2548,2024-09-15,England,Australia
498,36210,2628,2025-09-14,England,South Africa
593,40761,2725,2025-05-17,Royal Challengers Bengaluru,Kolkata Knight Riders


In [17]:

print("INNINGS AUDIT")
print("-------------")

innings_summary = (
    selected_ball
    .groupby("innings")
    .agg(
        Total_Deliveries=("match_id", "size"),
        Matches=("match_id", "nunique")
    )
    .reset_index()
)

display(innings_summary)

# Identify matches with third or fourth innings
extra_innings = selected_ball[
    selected_ball["innings"] > 2
].groupby(["match_id", "innings"]).size().reset_index(
    name="Delivery_Count"
)

print("Matches with innings beyond the second innings:")
display(extra_innings)


INNINGS AUDIT
-------------


,innings,Total_Deliveries,Matches
0,1,102054,837
1,2,91934,815
2,3,19,4
3,4,7,4


Matches with innings beyond the second innings:


,match_id,innings,Delivery_Count
0,34429,3,4
1,34429,4,1
2,40725,3,6
3,40725,4,4
4,45512,3,6
5,45512,4,1
6,50359,3,3
7,50359,4,1


In [18]:

print("DELIVERY KEY AUDIT")
print("------------------")

delivery_key = ["match_id", "innings", "innings_ball_num"]

print(
    "Missing key rows:",
    selected_ball[delivery_key].isna().any(axis=1).sum()
)

print(
    "Duplicate delivery keys:",
    selected_ball.duplicated(subset=delivery_key).sum()
)

# Date validation
selected_matches["match_date"] = pd.to_datetime(
    selected_matches["match_date"],
    errors="coerce"
)

print("\nDATE RANGE")
print("----------")
print("Earliest:", selected_matches["match_date"].min())
print("Latest:", selected_matches["match_date"].max())
print(
    "Dates before 2022:",
    (selected_matches["match_date"].dt.year < 2022).sum()
)


DELIVERY KEY AUDIT
------------------
Missing key rows: 0
Duplicate delivery keys: 0

DATE RANGE
----------
Earliest: 2022-01-22 00:00:00
Latest: 2026-11-01 00:00:00
Dates before 2022: 0


In [19]:

# A. Identify matches without a second innings
innings_counts = (
    selected_ball.groupby("match_id")["innings"]
    .nunique()
)

matches_without_second_innings = innings_counts[
    ~innings_counts.index.isin(
        selected_ball.loc[
            selected_ball["innings"] == 2, "match_id"
        ].unique()
    )
].index

print("MATCHES WITHOUT SECOND INNINGS")
display(
    selected_matches[
        selected_matches["match_id"].isin(
            matches_without_second_innings
        )
    ][[
        "match_id", "match_date", "home_team",
        "away_team", "result"
    ]]
)

# B. Inspect matches with third/fourth innings
extra_innings_ids = extra_innings["match_id"].unique()

print("\nMATCHES WITH EXTRA INNINGS")
display(
    selected_matches[
        selected_matches["match_id"].isin(extra_innings_ids)
    ][[
        "match_id", "match_date", "home_team",
        "away_team", "result"
    ]]
)

# C. Inspect all ball-by-ball column names
print("\nBALL-BY-BALL COLUMNS")
print(selected_ball.columns.tolist())


MATCHES WITHOUT SECOND INNINGS


,match_id,match_date,home_team,away_team,result
94,15410,2022-06-19,India,South Africa,Match Abandoned
95,15546,2022-07-02,West Indies,Bangladesh,No Result
234,20226,2023-05-03,Lucknow Super Giants,Chennai Super Kings,Match Abandoned
281,21406,2023-04-20,Pakistan,New Zealand,Match Abandoned
315,28333,2023-12-29,New Zealand,Bangladesh,No Result
420,31707,2024-04-18,Pakistan,New Zealand,No Result
464,37353,2024-11-18,West Indies,England,No Result
488,35424,2024-08-24,Canada,United States Of America,No Result
580,40748,2025-05-05,Sunrisers Hyderabad,Delhi Capitals,No Result
583,40751,2025-05-08,Punjab Kings,Delhi Capitals,No Result



MATCHES WITH EXTRA INNINGS


,match_id,match_date,home_team,away_team,result
478,34429,2024-07-30,Sri Lanka,India,India Tied with Srilanka (India Won the Super ...
557,40725,2025-04-16,Delhi Capitals,Rajasthan Royals,Delhi Capitals Tied with Rajasthan Royals (DC ...
679,45512,2025-09-26,India,Sri Lanka,India won the Super Over(Match Tied)
785,50359,2026-04-26,Lucknow Super Giants,Kolkata Knight Riders,Kolkata Knight Riders Won by Super over (Match...



BALL-BY-BALL COLUMNS
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']


In [20]:

# Inspect deliveries containing extras
extras_sample = selected_ball[
    selected_ball["extras_type"].notna()
]

print("Total deliveries with extras:", len(extras_sample))

print("\nExtras types:")
display(
    extras_sample["extras_type"].value_counts(dropna=False)
)

# Inspect relevant fields
relevant_columns = [
    col for col in [
        "match_id",
        "innings",
        "overid",
        "innings_ball_num",
        "striker_id",
        "bowler_id",
        "runs",
        "is_wicket",
        "dismissal_type",
        "dismissed_player_id",
        "extras_type",
        "extras1_runs",
        "extras_2_runs"
    ]
    if col in selected_ball.columns
]

display(extras_sample[relevant_columns].head(20))


Total deliveries with extras: 11466

Extras types:


extras_type
wide        7474
legbyes     2661
Overstep     565
byes         505
other        260
Penalty        1
Name: count, dtype: int64

,match_id,innings,overid,innings_ball_num,striker_id,bowler_id,runs,is_wicket,dismissal_type,dismissed_player_id,extras_type,extras1_runs,extras_2_runs
39,13511,1,7,40,478,2417,0,False,NaN,NaN,wide,2,0
43,13511,1,8,44,478,6726,0,False,NaN,NaN,wide,2,0
44,13511,1,8,45,478,6726,0,False,NaN,NaN,wide,2,0
143,13511,2,4,23,3639,479,0,False,NaN,NaN,legbyes,3,0
158,13511,2,7,38,4369,722,0,False,NaN,NaN,legbyes,3,0
278,13512,1,10,55,1596,494,0,False,NaN,NaN,wide,2,0
281,13512,1,10,58,1596,494,0,False,NaN,NaN,wide,2,0
292,13512,1,12,69,1596,2633,0,False,NaN,NaN,wide,2,0
307,13512,1,14,84,1924,494,0,False,NaN,NaN,wide,2,0
320,13512,1,16,97,478,494,0,False,NaN,NaN,wide,2,0


In [21]:

# STEP 1: Inspect relevant column names
print("RUN, EXTRA, WICKET AND BALL-RELATED COLUMNS")

relevant_names = [
    col for col in selected_ball.columns
    if any(word in col.lower() for word in [
        "run", "extra", "wicket", "dismiss",
        "over", "ball", "target", "wide", "noball"
    ])
]

print(relevant_names)


# STEP 2: Inspect data types
print("\nDATA TYPES")
display(
    selected_ball[relevant_names].dtypes.to_frame("data_type")
)


# STEP 3: Summary of runs and extras
run_columns = [
    col for col in [
        "runs", "extras1_runs", "extras_2_runs"
    ]
    if col in selected_ball.columns
]

print("\nRUN COLUMN SUMMARY")
display(selected_ball[run_columns].describe())


# STEP 4: Check extras by type
print("\nEXTRAS BREAKDOWN")
extras_summary = (
    selected_ball[
        selected_ball["extras_type"].notna()
    ]
    .groupby("extras_type")
    .agg(
        deliveries=("match_id", "size"),
        runs_column_total=("runs", "sum"),
        extras1_total=("extras1_runs", "sum"),
        extras2_total=("extras_2_runs", "sum")
    )
    .reset_index()
)

display(extras_summary)


# STEP 5: Inspect rows where extras_2_runs is non-zero
print("\nNON-ZERO extras_2_runs")
display(
    selected_ball[
        selected_ball["extras_2_runs"].fillna(0) != 0
    ][
        [
            col for col in [
                "match_id", "innings", "overid",
                "innings_ball_num", "runs",
                "extras_type", "extras1_runs",
                "extras_2_runs", "is_wicket",
                "dismissal_type"
            ]
            if col in selected_ball.columns
        ]
    ].head(30)
)


# STEP 6: Inspect non-zero runs on extras deliveries
print("\nEXTRAS DELIVERIES WITH NON-ZERO runs")
display(
    selected_ball[
        selected_ball["extras_type"].notna()
        & (selected_ball["runs"].fillna(0) != 0)
    ][
        [
            col for col in [
                "match_id", "innings", "overid",
                "innings_ball_num", "runs",
                "extras_type", "extras1_runs",
                "extras_2_runs"
            ]
            if col in selected_ball.columns
        ]
    ].head(30)
)


RUN, EXTRA, WICKET AND BALL-RELATED COLUMNS
['overid', 'over_ball_num', 'innings_ball_num', 'runs', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'over_or_around']

DATA TYPES


,data_type
overid,int64
over_ball_num,int64
innings_ball_num,int64
runs,int64
is_wicket,bool
dismissal_type_id,float64
dismissal_type,str
dismissed_player_id,float64
extras_type,str
extras1_runs,int64



RUN COLUMN SUMMARY


,runs,extras1_runs,extras_2_runs
count,194014.000000,194014.000000,194014.000000
mean,1.345877,5.783660,0.000428
std,1.732449,0.881799,0.037646
min,0.000000,1.000000,0.000000
25%,0.000000,6.000000,0.000000
50%,1.000000,6.000000,0.000000
75%,1.000000,6.000000,0.000000
max,6.000000,7.000000,4.000000



EXTRAS BREAKDOWN


,extras_type,deliveries,runs_column_total,extras1_total,extras2_total
0,Overstep,565,834,565,48
1,Penalty,1,0,7,0
2,byes,505,0,2020,0
3,legbyes,2661,0,7983,0
4,other,260,411,1300,34
5,wide,7474,0,14948,0



NON-ZERO extras_2_runs


,match_id,innings,overid,innings_ball_num,runs,extras_type,extras1_runs,extras_2_runs,is_wicket,dismissal_type
1485,13523,1,1,4,0,Overstep,1,4,False,NaN
2588,13614,1,18,111,0,other,5,4,False,NaN
7607,14112,1,11,66,0,Overstep,1,4,False,NaN
11386,14127,2,9,52,0,other,5,4,False,NaN
18844,14159,2,9,51,0,other,5,1,False,NaN
24453,15796,2,14,88,0,Overstep,1,1,False,NaN
28010,16424,2,6,33,0,Overstep,1,4,False,NaN
36854,17555,1,5,29,0,NaN,6,1,False,NaN
47440,20186,2,20,130,0,Overstep,1,1,False,NaN
47803,20188,1,19,117,0,other,5,1,False,NaN



EXTRAS DELIVERIES WITH NON-ZERO runs


,match_id,innings,overid,innings_ball_num,runs,extras_type,extras1_runs,extras_2_runs
1226,13515,2,19,119,2,other,5,0
1229,13515,2,20,122,1,Overstep,1,0
1486,13523,1,1,5,1,Overstep,1,0
3309,14094,1,19,119,6,other,5,0
4187,14098,1,4,23,4,Overstep,1,0
4192,14098,1,5,28,1,Overstep,1,0
4610,14099,2,13,76,1,Overstep,1,0
5099,14101,2,12,74,1,Overstep,1,0
5111,14101,2,14,86,4,Overstep,1,0
5265,14102,2,4,22,4,Overstep,1,0


In [22]:

# STEP 1: Display every column name
for i, col in enumerate(selected_ball.columns):
    print(i, col)


# STEP 2: Inspect all rows for one match with different extras
sample_match = selected_ball[
    selected_ball["match_id"] == 13523
].copy()

columns_to_show = [
    col for col in [
        "match_id",
        "innings",
        "overid",
        "innings_ball_num",
        "striker_id",
        "bowler_id",
        "runs",
        "extras_type",
        "extras1_runs",
        "extras_2_runs",
        "is_wicket",
        "dismissal_type"
    ]
    if col in sample_match.columns
]

display(sample_match[columns_to_show].head(25))


# STEP 3: Check unique values in the two extras columns
print("extras1_runs unique values:")
print(sorted(selected_ball["extras1_runs"].dropna().unique()))

print("\nextras_2_runs unique values:")
print(sorted(selected_ball["extras_2_runs"].dropna().unique()))


# STEP 4: Check deliveries with no extras
print("\nDELIVERIES WITHOUT EXTRAS")
display(
    selected_ball[
        selected_ball["extras_type"].isna()
    ][columns_to_show].head(15)
)


0 match_id
1 innings
2 overid
3 over_ball_num
4 innings_ball_num
5 batting_team_id
6 bowling_team_id
7 striker_id
8 striker_name
9 non_striker_id
10 non_striker_name
11 bowler_id
12 bowler_name
13 batting_type
14 bowling_type
15 batting_order
16 runs
17 is_four
18 is_six
19 is_wicket
20 dismissal_type_id
21 dismissal_type
22 dismissed_player_id
23 extras_type
24 extras1_runs
25 extras_2_runs
26 pace_or_spin
27 bowl_speed
28 stroke
29 delivery_style
30 over_or_around
31 on_or_off
32 delivery_ist_time


,match_id,innings,overid,innings_ball_num,striker_id,bowler_id,runs,extras_type,extras1_runs,extras_2_runs,is_wicket,dismissal_type
1482,13523,1,1,1,126,2417,1,NaN,6,0,False,NaN
1483,13523,1,1,2,4117,2417,0,NaN,6,0,False,NaN
1484,13523,1,1,3,4117,2417,0,NaN,6,0,False,NaN
1485,13523,1,1,4,4117,2417,0,Overstep,1,4,False,NaN
1486,13523,1,1,5,4117,2417,1,Overstep,1,0,False,NaN
1487,13523,1,1,6,126,2417,1,NaN,6,0,False,NaN
1488,13523,1,1,7,4117,2417,1,NaN,6,0,False,NaN
1489,13523,1,1,8,126,2417,0,NaN,6,0,False,NaN
1490,13523,1,2,9,4117,1772,0,NaN,6,0,False,NaN
1491,13523,1,2,10,4117,1772,0,NaN,6,0,False,NaN


extras1_runs unique values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]

extras_2_runs unique values:
[np.int64(0), np.int64(1), np.int64(4)]

DELIVERIES WITHOUT EXTRAS


,match_id,innings,overid,innings_ball_num,striker_id,bowler_id,runs,extras_type,extras1_runs,extras_2_runs,is_wicket,dismissal_type
0,13511,1,1,1,1187,1772,0,NaN,6,0,False,NaN
1,13511,1,1,2,1187,1772,0,NaN,6,0,False,NaN
2,13511,1,1,3,1187,1772,0,NaN,6,0,False,NaN
3,13511,1,1,4,1187,1772,0,NaN,6,0,False,NaN
4,13511,1,1,5,1187,1772,6,NaN,6,0,False,NaN
5,13511,1,1,6,1187,1772,0,NaN,6,0,True,Bowled
6,13511,1,2,7,6273,2633,0,NaN,6,0,False,NaN
7,13511,1,2,8,6273,2633,0,NaN,6,0,False,NaN
8,13511,1,2,9,6273,2633,0,NaN,6,0,False,NaN
9,13511,1,2,10,6273,2633,4,NaN,6,0,False,NaN


In [23]:
# STEP 1: Inspect all deliveries that contain extras
extras_sample = selected_ball[
    selected_ball["extras_type"].notna()
].copy()

columns_to_show = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "innings_ball_num",
    "striker_name",
    "bowler_name",
    "runs",
    "extras_type",
    "extras1_runs",
    "extras_2_runs",
    "is_wicket",
    "dismissal_type"
]

display(extras_sample[columns_to_show].head(40))


# STEP 2: Check combinations of extras type, runs and extras_2_runs
print("EXTRAS TYPE + RUNS + EXTRAS_2_RUNS COMBINATIONS")

extras_combinations = (
    selected_ball
    .groupby(
        ["extras_type", "runs", "extras_2_runs"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
    .sort_values(
        ["extras_type", "runs", "extras_2_runs"],
        na_position="first"
    )
)

display(extras_combinations)


# STEP 3: Inspect over-ball numbering
print("OVER BALL NUMBER DISTRIBUTION")

display(
    selected_ball["over_ball_num"]
    .value_counts(dropna=False)
    .sort_index()
)


# STEP 4: Inspect all unique dismissal types
print("DISMISSAL TYPES")

print(
    selected_ball["dismissal_type"]
    .value_counts(dropna=False)
)

,match_id,innings,overid,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,extras_type,extras1_runs,extras_2_runs,is_wicket,dismissal_type
39,13511,1,7,4,40,Eoin Morgan,Akeal Hosein,0,wide,2,0,False,NaN
43,13511,1,8,1,44,Eoin Morgan,Romario Shepherd,0,wide,2,0,False,NaN
44,13511,1,8,2,45,Eoin Morgan,Romario Shepherd,0,wide,2,0,False,NaN
143,13511,2,4,5,23,Brandon King,Adil Rashid,0,legbyes,3,0,False,NaN
158,13511,2,7,2,38,Shai Hope,Liam Dawson,0,legbyes,3,0,False,NaN
278,13512,1,10,1,55,Moeen Ali,Kieron Pollard,0,wide,2,0,False,NaN
281,13512,1,10,4,58,Moeen Ali,Kieron Pollard,0,wide,2,0,False,NaN
292,13512,1,12,1,69,Moeen Ali,Jason Holder,0,wide,2,0,False,NaN
307,13512,1,14,3,84,Sam Billings,Kieron Pollard,0,wide,2,0,False,NaN
320,13512,1,16,3,97,Eoin Morgan,Kieron Pollard,0,wide,2,0,False,NaN


EXTRAS TYPE + RUNS + EXTRAS_2_RUNS COMBINATIONS


,extras_type,runs,extras_2_runs,delivery_count
20,NaN,0,0,65460
21,NaN,0,1,1
22,NaN,1,0,69412
23,NaN,2,0,12132
24,NaN,3,0,635
25,NaN,4,0,22566
26,NaN,5,0,23
27,NaN,6,0,12319
0,Overstep,0,0,215
1,Overstep,0,1,12


OVER BALL NUMBER DISTRIBUTION


over_ball_num
1     31217
2     31126
3     31044
4     30944
5     30820
6     30680
7      6500
8      1333
9       275
10       56
11       15
12        2
13        2
Name: count, dtype: int64

DISMISSAL TYPES
dismissal_type
NaN                      183528
Caught                     7332
Bowled                     1690
LBW                         666
RunOut                      573
Stump                       200
Hit Wicket                   17
Retire Out                    6
Obstracting the field         1
Obstructing the field         1
Name: count, dtype: int64


In [24]:
# STEP 1: Inspect every delivery where extras_2_runs is non-zero

exceptional_extras = selected_ball[
    selected_ball["extras_2_runs"] > 0
].copy()

display(
    exceptional_extras[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].to_string(index=False)
)


# STEP 2: Inspect match score-related columns

print("MATCH DATASET COLUMNS")

for col in selected_matches.columns:
    print(col)


# STEP 3: Inspect all extras categories and their frequency

print("\nEXTRAS CATEGORY COUNTS")

display(
    selected_ball["extras_type"]
    .value_counts(dropna=False)
)


# STEP 4: Check whether over_ball_num resets after each over

sample = selected_ball[
    (selected_ball["match_id"] == 13523) &
    (selected_ball["innings"] == 1)
]

display(
    sample[
        [
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "runs",
            "extras_type",
            "extras_2_runs"
        ]
    ].head(40)
)

' match_id  innings  overid  over_ball_num  innings_ball_num           striker_name         bowler_name  runs extras_type  extras1_runs  extras_2_runs  is_wicket dismissal_type\n    13523        1       1              4                 4           Ishan Kishan        Akeal Hosein     0    Overstep             1              4      False            NaN\n    13614        1      18              5               111          Dasun Shanaka      Mitchell Starc     0       other             5              4      False            NaN\n    14112        1      11              5                66           David Warner  Varun Chakravarthy     0    Overstep             1              4      False            NaN\n    14127        2       9              3                52           Prithvi Shaw          Obed McCoy     0       other             5              4      False            NaN\n    14159        2       9              1                51           Ishan Kishan         Umran Malik     0      

MATCH DATASET COLUMNS
match_id
comp_id
match_number
match_date
stage
venue_id
venue_country
home_team_id
home_team
away_team_id
away_team
home_team_captain_id
away_team_captain_id
toss_win_team_id
toss_opted
man_of_the_match
first_inn_revised_overs
second_inn_revised_overs
revised_target
knock_out
win_team_id
result

EXTRAS CATEGORY COUNTS


extras_type
NaN         182548
wide          7474
legbyes       2661
Overstep       565
byes           505
other          260
Penalty          1
Name: count, dtype: int64

,overid,over_ball_num,innings_ball_num,runs,extras_type,extras_2_runs
1482,1,1,1,1,NaN,0
1483,1,2,2,0,NaN,0
1484,1,3,3,0,NaN,0
1485,1,4,4,0,Overstep,4
1486,1,5,5,1,Overstep,0
1487,1,6,6,1,NaN,0
1488,1,7,7,1,NaN,0
1489,1,8,8,0,NaN,0
1490,2,1,9,0,NaN,0
1491,2,2,10,0,NaN,0


In [25]:
# Inspect exceptional extras records only

exceptional_extras = selected_ball[
    selected_ball["extras_2_runs"] > 0
].copy()

print("Number of exceptional deliveries:", len(exceptional_extras))

display(
    exceptional_extras[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].to_string(index=False)
)

print("\nCounts by extras type and extras_2_runs:")

display(
    exceptional_extras.groupby(
        ["extras_type", "extras_2_runs"],
        dropna=False
    ).size().reset_index(name="count")
)

Number of exceptional deliveries: 35


' match_id  innings  overid  over_ball_num           striker_name         bowler_name  runs extras_type  extras1_runs  extras_2_runs  is_wicket dismissal_type\n    13523        1       1              4           Ishan Kishan        Akeal Hosein     0    Overstep             1              4      False            NaN\n    13614        1      18              5          Dasun Shanaka      Mitchell Starc     0       other             5              4      False            NaN\n    14112        1      11              5           David Warner  Varun Chakravarthy     0    Overstep             1              4      False            NaN\n    14127        2       9              3           Prithvi Shaw          Obed McCoy     0       other             5              4      False            NaN\n    14159        2       9              1           Ishan Kishan         Umran Malik     0       other             5              1      False            NaN\n    15796        2      14              4    


Counts by extras type and extras_2_runs:


,extras_type,extras_2_runs,count
0,Overstep,1,12
1,Overstep,4,9
2,other,1,6
3,other,4,7
4,NaN,1,1


In [26]:
# Display every delivery where extras_2_runs is greater than zero

exceptional_extras = selected_ball[
    selected_ball["extras_2_runs"] > 0
].copy()

display(
    exceptional_extras[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].to_string(index=False)
)

' match_id  innings  overid  over_ball_num           striker_name         bowler_name  runs extras_type  extras1_runs  extras_2_runs  is_wicket dismissal_type\n    13523        1       1              4           Ishan Kishan        Akeal Hosein     0    Overstep             1              4      False            NaN\n    13614        1      18              5          Dasun Shanaka      Mitchell Starc     0       other             5              4      False            NaN\n    14112        1      11              5           David Warner  Varun Chakravarthy     0    Overstep             1              4      False            NaN\n    14127        2       9              3           Prithvi Shaw          Obed McCoy     0       other             5              4      False            NaN\n    14159        2       9              1           Ishan Kishan         Umran Malik     0       other             5              1      False            NaN\n    15796        2      14              4    

In [28]:
print("Unique extras1_runs values:")
print(sorted(selected_ball["extras1_runs"].dropna().unique()))

print("\nUnique extras_2_runs values:")
print(sorted(selected_ball["extras_2_runs"].dropna().unique()))

Unique extras1_runs values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]

Unique extras_2_runs values:
[np.int64(0), np.int64(1), np.int64(4)]


In [29]:
# Inspect representative exceptional deliveries
sample_ids = [13523, 13614, 17555, 20201, 20243]

display(
    selected_ball[
        selected_ball["match_id"].isin(sample_ids)
    ].sort_values(
        ["match_id", "innings", "overid", "over_ball_num"]
    ).to_string(index=False)
)

' match_id  innings  overid  over_ball_num  innings_ball_num  batting_team_id  bowling_team_id  striker_id         striker_name  non_striker_id     non_striker_name  bowler_id         bowler_name batting_type           bowling_type  batting_order  runs  is_four  is_six  is_wicket  dismissal_type_id dismissal_type  dismissed_player_id extras_type  extras1_runs  extras_2_runs pace_or_spin  bowl_speed              stroke    delivery_style over_or_around on_or_off       delivery_ist_time\n    13523        1       1              1                 1               18               28         126         Rohit Sharma            4117         Ishan Kishan       2417        Akeal Hosein          RHB Slow left-arm orthodox              1     1    False   False      False                NaN            NaN                  NaN         NaN             6              0         Spin        0.00 Frontfoot defensive          Arm ball         Around   On-side 2022-02-18 19:00:32 UTC\n    13523        1   

In [30]:
display(
    selected_matches[
        selected_matches["match_id"].isin(sample_ids)
    ].to_string(index=False)
)

' match_id  comp_id  match_number match_date  stage  venue_id venue_country  home_team_id            home_team  away_team_id      away_team  home_team_captain_id  away_team_captain_id  toss_win_team_id toss_opted  man_of_the_match  first_inn_revised_overs  second_inn_revised_overs  revised_target  knock_out  win_team_id                             result\n    13523     1335             2 2022-02-18 League         8         India            18                India            28    West Indies                 126.0                 494.0              28.0   Fielding                 0                      NaN                       NaN             NaN          0         18.0                India Won by 8 Runs\n    13614     1345             3 2022-02-15 League       129     Australia            20            Australia            27      Sri Lanka                   NaN                2436.0              20.0   Fielding               286                      NaN                       NaN     

In [31]:
# Inspect deliveries around two exceptional records

examples = [
    (20201, 1, 11),  # IPL: Overstep
    (13523, 1, 1)    # T20I: Overstep
]

for match_id, innings, overid in examples:

    print(f"\nMATCH: {match_id} | INNINGS: {innings} | OVER: {overid}")

    display(
        selected_ball[
            (selected_ball["match_id"] == match_id) &
            (selected_ball["innings"] == innings) &
            (selected_ball["overid"] == overid)
        ][
            [
                "over_ball_num",
                "striker_name",
                "bowler_name",
                "runs",
                "extras_type",
                "extras1_runs",
                "extras_2_runs",
                "is_four",
                "is_six",
                "is_wicket"
            ]
        ].to_string(index=False)
    )


MATCH: 20201 | INNINGS: 1 | OVER: 11


' over_ball_num  striker_name   bowler_name  runs extras_type  extras1_runs  extras_2_runs  is_four  is_six  is_wicket\n             1 Krunal Pandya Sikandar Raza     1         NaN             6              0    False   False      False\n             2  Lokesh Rahul Sikandar Raza     2         NaN             6              0    False   False      False\n             3  Lokesh Rahul Sikandar Raza     1         NaN             6              0    False   False      False\n             4 Krunal Pandya Sikandar Raza     0         NaN             6              0    False   False      False\n             5 Krunal Pandya Sikandar Raza     0    Overstep             1              4    False   False      False\n             6 Krunal Pandya Sikandar Raza     4         NaN             6              0     True   False      False\n             7 Krunal Pandya Sikandar Raza     1         NaN             6              0    False   False      False'


MATCH: 13523 | INNINGS: 1 | OVER: 1


' over_ball_num striker_name  bowler_name  runs extras_type  extras1_runs  extras_2_runs  is_four  is_six  is_wicket\n             1 Rohit Sharma Akeal Hosein     1         NaN             6              0    False   False      False\n             2 Ishan Kishan Akeal Hosein     0         NaN             6              0    False   False      False\n             3 Ishan Kishan Akeal Hosein     0         NaN             6              0    False   False      False\n             4 Ishan Kishan Akeal Hosein     0    Overstep             1              4    False   False      False\n             5 Ishan Kishan Akeal Hosein     1    Overstep             1              0    False   False      False\n             6 Rohit Sharma Akeal Hosein     1         NaN             6              0    False   False      False\n             7 Ishan Kishan Akeal Hosein     1         NaN             6              0    False   False      False\n             8 Rohit Sharma Akeal Hosein     0         NaN     

In [32]:
# Check the relationship between extras_type and extras1_runs

extras_mapping = (
    selected_ball
    .groupby(
        ["extras_type", "extras1_runs"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
    .sort_values(["extras1_runs", "extras_type"])
)

display(extras_mapping)

print("\nRelationship between extras_type and extras_2_runs:")

extras_2_mapping = (
    selected_ball
    .groupby(
        ["extras_type", "extras_2_runs"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
    .sort_values(["extras_type", "extras_2_runs"])
)

display(extras_2_mapping)

,extras_type,extras1_runs,delivery_count
0,Overstep,1,565
5,wide,2,7474
3,legbyes,3,2661
2,byes,4,505
4,other,5,260
6,NaN,6,182548
1,Penalty,7,1



Relationship between extras_type and extras_2_runs:


,extras_type,extras_2_runs,delivery_count
0,Overstep,0,544
1,Overstep,1,12
2,Overstep,4,9
3,Penalty,0,1
4,byes,0,505
5,legbyes,0,2661
6,other,0,247
7,other,1,6
8,other,4,7
9,wide,0,7474


In [34]:
# Audit batter-run values against boundary indicators

print("RUNS DISTRIBUTION")
display(
    selected_ball["runs"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nFOUR INDICATOR CHECK")
display(
    selected_ball.groupby("is_four", dropna=False)["runs"]
    .value_counts(dropna=False)
)

print("\nSIX INDICATOR CHECK")
display(
    selected_ball.groupby("is_six", dropna=False)["runs"]
    .value_counts(dropna=False)
)

print("\nINCONSISTENT BOUNDARY RECORDS")

boundary_errors = selected_ball[
    ((selected_ball["is_four"] == True) & (selected_ball["runs"] != 4)) |
    ((selected_ball["is_six"] == True) & (selected_ball["runs"] != 6))
]

display(
    boundary_errors[
        [
            "match_id", "innings", "overid",
            "striker_name", "runs", "extras_type",
            "is_four", "is_six"
        ]
    ].head(20)
)

print("Number of inconsistent boundary records:", len(boundary_errors))

RUNS DISTRIBUTION


runs
0    76463
1    69628
2    12183
3      638
4    22689
5       23
6    12390
Name: count, dtype: int64


FOUR INDICATOR CHECK


is_four  runs
False    0       76463
         1       69628
         6       12390
         2       12183
         3         638
         5          23
         4          20
True     4       22669
Name: count, dtype: int64


SIX INDICATOR CHECK


is_six  runs
False   0       76463
        1       69628
        4       22689
        2       12183
        3         638
        5          23
        6           3
True    6       12387
Name: count, dtype: int64


INCONSISTENT BOUNDARY RECORDS


,match_id,innings,overid,striker_name,runs,extras_type,is_four,is_six


Number of inconsistent boundary records: 0


In [35]:
# Inspect unusual four- and six-run deliveries

unusual_boundaries = selected_ball[
    ((selected_ball["runs"] == 4) & (selected_ball["is_four"] == False)) |
    ((selected_ball["runs"] == 6) & (selected_ball["is_six"] == False))
]

display(
    unusual_boundaries[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "is_four",
            "is_six",
            "stroke",
            "delivery_style"
        ]
    ].to_string(index=False)
)

' match_id  innings  overid  over_ball_num           striker_name         bowler_name  runs extras_type  is_four  is_six              stroke delivery_style\n    14113        1       1              3       Devdutt Padikkal Dushmantha Chameera     4         NaN    False   False         Cover Drive      Out Swing\n    14123        1       3              6            Jos Buttler         Umesh Yadav     4         NaN    False   False        Square drive         Yorker\n    14149        1      12              2           Devon Conway      Mitchell Marsh     4         NaN    False   False           Pull shot     Seamed Out\n    15609        2      17              8      Najibullah Zadran Blessing Muzarabani     4         NaN    False   False           Pull shot     Off Cutter\n    16425        2      13              1           Afif Hossain    Wesley Madhevere     4         NaN    False   False              Worked      Off Break\n    16754        1       6              1      Najibullah Zadra

In [37]:
# Investigate unusual boundary records

print("Unusual boundary records by runs:")

display(
    unusual_boundaries.groupby(
        ["runs", "extras_type", "is_four", "is_six"],
        dropna=False
    ).size().reset_index(name="count")
)

print("\nCheck for unusual boundary records with extras:")

print(
    "Records with extras:",
    unusual_boundaries["extras_type"].notna().sum()
)

print(
    "Records with wickets:",
    unusual_boundaries["is_wicket"].sum()
)

print(
    "Unique runs:",
    sorted(unusual_boundaries["runs"].unique())
)

Unusual boundary records by runs:


,runs,extras_type,is_four,is_six,count
0,4,NaN,False,False,20
1,6,NaN,False,False,3



Check for unusual boundary records with extras:
Records with extras: 0
Records with wickets: 0
Unique runs: [np.int64(4), np.int64(6)]


In [38]:
# Derived boundary indicators
selected_ball["derived_four"] = (
    selected_ball["runs"] == 4
)

selected_ball["derived_six"] = (
    selected_ball["runs"] == 6
)

In [39]:
# Inspect complete innings summary for match 20201

match_id = 20201

innings_data = selected_ball[
    (selected_ball["match_id"] == match_id) &
    (selected_ball["innings"] == 1)
].copy()

print("Total recorded deliveries:", len(innings_data))
print("Total runs in runs column:", innings_data["runs"].sum())

print("\nExtras breakdown:")

display(
    innings_data.groupby(
        "extras_type",
        dropna=False
    ).agg(
        deliveries=("runs", "size"),
        runs_sum=("runs", "sum")
    ).reset_index()
)

print("\nWickets recorded:", innings_data["is_wicket"].sum())

print("\nLast 10 deliveries:")

display(
    innings_data.tail(10)[
        [
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "is_wicket"
        ]
    ]
)

Total recorded deliveries: 127
Total runs in runs column: 147

Extras breakdown:


,extras_type,deliveries,runs_sum
0,Overstep,1,0
1,legbyes,1,0
2,wide,6,0
3,NaN,119,147



Wickets recorded: 8

Last 10 deliveries:


,overid,over_ball_num,striker_name,bowler_name,runs,extras_type,is_wicket
50987,19,4,Lokesh Rahul,Arshdeep Singh,0,NaN,True
50988,19,5,Krishnappa Gowtham,Arshdeep Singh,1,NaN,False
50989,19,6,Ayush Badoni,Arshdeep Singh,0,legbyes,False
50990,20,1,Ayush Badoni,Sam Curran,0,wide,False
50991,20,2,Ayush Badoni,Sam Curran,0,NaN,False
50992,20,3,Ayush Badoni,Sam Curran,1,NaN,False
50993,20,4,Krishnappa Gowtham,Sam Curran,0,NaN,True
50994,20,5,Yudhvir Singh,Sam Curran,0,NaN,True
50995,20,6,Ravi Bishnoi,Sam Curran,3,NaN,False
50996,20,7,Ayush Badoni,Sam Curran,2,NaN,False


In [40]:
# Inspect all extras deliveries in match 20201, innings 1

extras_check = innings_data[
    innings_data["extras_type"].notna()
].copy()

columns_to_check = [
    "overid",
    "over_ball_num",
    "striker_name",
    "bowler_name",
    "runs",
    "extras_type",
    "extras1_runs",
    "extras_2_runs",
    "is_wicket"
]

display(extras_check[columns_to_check].to_string(index=False))

print("\nExtras field totals:")
print("Number of extras records:", len(extras_check))
print("Sum of extras_2_runs:", extras_check["extras_2_runs"].sum())
print("Sum of runs on extras records:", extras_check["runs"].sum())

' overid  over_ball_num    striker_name    bowler_name  runs extras_type  extras1_runs  extras_2_runs  is_wicket\n      2              1    Lokesh Rahul Arshdeep Singh     0        wide             2              0      False\n     11              5   Krunal Pandya  Sikandar Raza     0    Overstep             1              4      False\n     14              1    Lokesh Rahul   Rahul Chahar     0        wide             2              0      False\n     15              3 Nicholas Pooran  Kagiso Rabada     0        wide             2              0      False\n     17              5    Lokesh Rahul  Kagiso Rabada     0        wide             2              0      False\n     18              4    Lokesh Rahul     Sam Curran     0        wide             2              0      False\n     19              6    Ayush Badoni Arshdeep Singh     0     legbyes             3              0      False\n     20              1    Ayush Badoni     Sam Curran     0        wide             2          


Extras field totals:
Number of extras records: 8
Sum of extras_2_runs: 4
Sum of runs on extras records: 0


In [41]:
# Check the complete source columns for extras deliveries

display(
    extras_check.to_string(index=False)
)

' match_id  innings  overid  over_ball_num  innings_ball_num  batting_team_id  bowling_team_id  striker_id    striker_name  non_striker_id   non_striker_name  bowler_id    bowler_name batting_type         bowling_type  batting_order  runs  is_four  is_six  is_wicket  dismissal_type_id dismissal_type  dismissed_player_id extras_type  extras1_runs  extras_2_runs pace_or_spin  bowl_speed     stroke delivery_style over_or_around on_or_off       delivery_ist_time  derived_four  derived_six\n    20201        1       2              1                 7             3509                4         953    Lokesh Rahul            2416        Kyle Mayers       6464 Arshdeep Singh          RHB Left-arm medium-fast              1     0    False   False      False                NaN            NaN                  NaN        wide             2              0         Pace       134.8        NaN    Angled Away         Around  Off-side 2023-04-15 19:34:29 UTC         False        False\n    20201        1 

### Extras Field Audit

The ball-by-ball dataset contains three extras-related fields.

- extras_type: identifies the extras category.
- extras1_runs: categorical code for the extras category.
- extras_2_runs: secondary field with an unresolved meaning.

The runs column is consistent with batter runs in the validated innings.

Validation against LSG vs PBKS, IPL 2023, Match 21:
- Batter runs: 147
- Official extras: 12
- Official total: 159/8
- Dataset batter runs: 147
- Dataset extras records: 8

The dataset does not currently provide sufficient information
to reconstruct the actual extras-run amount for every delivery.

Therefore, extras_2_runs will not be used as a run value.
Raw source columns will remain unchanged.

In [42]:
# Summarize extras by category across all selected deliveries

extras_audit = (
    selected_ball
    .groupby("extras_type", dropna=False)
    .agg(
        delivery_count=("match_id", "size"),
        batter_runs=("runs", "sum"),
        secondary_field_total=("extras_2_runs", "sum"),
        unique_secondary_values=("extras_2_runs", "nunique")
    )
    .reset_index()
)

display(extras_audit)

,extras_type,delivery_count,batter_runs,secondary_field_total,unique_secondary_values
0,Overstep,565,834,48,3
1,Penalty,1,0,0,1
2,byes,505,0,0,1
3,legbyes,2661,0,0,1
4,other,260,411,34,3
5,wide,7474,0,0,1
6,NaN,182548,259874,1,2


In [43]:
# Inspect run values for unusual extras categories

unusual_extras = (
    selected_ball[
        selected_ball["extras_type"].isin(["Overstep", "other", "Penalty"])
    ]
    .groupby(
        ["extras_type", "runs", "extras_2_runs", "is_four", "is_six"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
    .sort_values(["extras_type", "runs"])
)

display(unusual_extras)

,extras_type,runs,extras_2_runs,is_four,is_six,delivery_count
0,Overstep,0,0,False,False,215
1,Overstep,0,1,False,False,12
2,Overstep,0,4,False,False,9
3,Overstep,1,0,False,False,168
4,Overstep,2,0,False,False,33
5,Overstep,3,0,False,False,2
6,Overstep,4,0,True,False,81
7,Overstep,6,0,False,True,45
8,Penalty,0,0,False,False,1
9,other,0,0,False,False,112


In [44]:
# Inspect actual deliveries with unusual extras and non-zero runs

sample_deliveries = (
    selected_ball[
        selected_ball["extras_type"].isin(["Overstep", "other"])
        & (selected_ball["runs"] > 0)
    ][
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "is_four",
            "is_six",
            "extras_type",
            "extras_2_runs",
            "stroke"
        ]
    ]
    .sort_values(["match_id", "innings", "overid", "over_ball_num"])
)

display(sample_deliveries.head(25))

,match_id,innings,overid,over_ball_num,striker_name,bowler_name,runs,is_four,is_six,extras_type,extras_2_runs,stroke
1226,13515,2,19,5,Sam Billings,Sheldon Cottrell,2,False,False,other,0,Pull shot
1229,13515,2,20,1,Sam Billings,Jason Holder,1,False,False,Overstep,0,On drive
1486,13523,1,1,5,Ishan Kishan,Akeal Hosein,1,False,False,Overstep,0,Flick
3309,14094,1,19,7,Mahendra Singh Dhoni,Shivam Mavi,6,False,True,other,0,Pull shot
4187,14098,1,4,3,Jos Buttler,Umran Malik,4,True,False,Overstep,0,Backfoot defensive
4192,14098,1,5,1,Yashasvi Jaiswal,Washington Sundar,1,False,False,Overstep,0,Worked
4610,14099,2,13,2,Shahbaz Ahmed,Andre Russell,1,False,False,Overstep,0,Cover Drive
5099,14101,2,12,6,Andre Russell,Odean Smith,1,False,False,Overstep,0,Pull shot
5111,14101,2,14,4,Andre Russell,Arshdeep Singh,4,True,False,Overstep,0,Straight Drive
5265,14102,2,4,3,Ishan Kishan,Navdeep Saini,4,True,False,Overstep,0,Off Drive


In [45]:
# Find innings with the most unusual extras deliveries

unusual_match_summary = (
    selected_ball[
        selected_ball["extras_type"].isin(["Overstep", "other"])
    ]
    .groupby(["match_id", "innings"])
    .agg(
        unusual_extras_count=("runs", "size"),
        batter_runs_on_unusual_extras=("runs", "sum")
    )
    .reset_index()
    .sort_values(
        "unusual_extras_count",
        ascending=False
    )
)

display(unusual_match_summary.head(10))

,match_id,innings,unusual_extras_count,batter_runs_on_unusual_extras
125,19557,1,7,21
337,36209,1,5,6
549,50368,1,4,6
258,30076,2,4,6
76,15796,2,4,6
95,16657,2,4,6
161,20218,1,4,7
191,20249,2,4,13
329,36096,1,4,5
7,14098,1,4,5


In [46]:
# Get match details for our validation candidate

match_to_validate = 19557

match_details = (
    selected_matches[
        selected_matches["match_id"] == match_to_validate
    ]
)

display(match_details.T)

,181
match_id,19557
comp_id,1698
match_number,2
match_date,2023-01-05 00:00:00
stage,League
venue_id,119
venue_country,India
home_team_id,18
home_team,India
away_team_id,27


In [47]:
# Summarise the selected innings

innings_1 = selected_ball[
    (selected_ball["match_id"] == 19557) &
    (selected_ball["innings"] == 1)
]

print("Delivery records:", len(innings_1))
print("Sum of runs:", innings_1["runs"].sum())
print("Wickets recorded:", innings_1["is_wicket"].sum())

print("\nExtras breakdown:")
display(
    innings_1.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("runs", "size"),
        runs_total=("runs", "sum")
    )
    .reset_index()
)

Delivery records: 131
Sum of runs: 194
Wickets recorded: 6

Extras breakdown:


,extras_type,deliveries,runs_total
0,Overstep,7,21
1,legbyes,1,0
2,wide,4,0
3,NaN,119,173


In [48]:
# Create a validated working batter-runs field

selected_ball["batter_runs"] = selected_ball["runs"]

# Validation checks
print("Total delivery records:", len(selected_ball))
print("Total batter runs:", selected_ball["batter_runs"].sum())
print("Missing batter runs:", selected_ball["batter_runs"].isna().sum())

print("\nBatter runs distribution:")
display(
    selected_ball["batter_runs"].value_counts().sort_index()
)

Total delivery records: 194014
Total batter runs: 261119
Missing batter runs: 0

Batter runs distribution:


batter_runs
0    76463
1    69628
2    12183
3      638
4    22689
5       23
6    12390
Name: count, dtype: int64

In [49]:
# Audit delivery types before creating legal-ball flags

delivery_type_audit = (
    selected_ball
    .assign(
        delivery_type=selected_ball["extras_type"].fillna("normal")
    )
    .groupby("delivery_type")
    .agg(
        delivery_count=("match_id", "size"),
        batter_runs=("batter_runs", "sum"),
        wicket_count=("is_wicket", "sum")
    )
    .reset_index()
    .sort_values("delivery_count", ascending=False)
)

display(delivery_type_audit)

,delivery_type,delivery_count,batter_runs,wicket_count
4,normal,182548,259874,10439
6,wide,7474,0,39
3,legbyes,2661,0,0
0,Overstep,565,834,5
2,byes,505,0,1
5,other,260,411,2
1,Penalty,1,0,0


In [50]:
# Inspect wickets recorded on unusual delivery types

wicket_extras_audit = (
    selected_ball[
        (selected_ball["is_wicket"] == True) &
        (selected_ball["extras_type"].notna())
    ]
    .groupby(
        ["extras_type", "dismissal_type"],
        dropna=False
    )
    .size()
    .reset_index(name="wicket_count")
    .sort_values(
        ["extras_type", "wicket_count"],
        ascending=[True, False]
    )
)

display(wicket_extras_audit)

,extras_type,dismissal_type,wicket_count
0,Overstep,RunOut,5
1,byes,RunOut,1
2,other,RunOut,2
4,wide,Stump,25
3,wide,RunOut,14


In [51]:
# View actual wicket deliveries with extras

wicket_extras_samples = selected_ball[
    (selected_ball["is_wicket"] == True) &
    (selected_ball["extras_type"].notna())
][
    [
        "match_id",
        "innings",
        "overid",
        "striker_name",
        "bowler_name",
        "extras_type",
        "dismissal_type",
        "dismissed_player_id",
        "runs"
    ]
]

display(wicket_extras_samples.head(30))

,match_id,innings,overid,striker_name,bowler_name,extras_type,dismissal_type,dismissed_player_id,runs
3241,14094,1,8,Robin Uthappa,Varun Chakravarthy,wide,Stump,169.0,0
5365,14102,2,19,Kieron Pollard,Prasidh Krishna,other,RunOut,5034.0,0
15456,14145,1,19,Namboori Tilak Varma,Lockie Ferguson,other,RunOut,7897.0,0
20026,14164,1,20,Romario Shepherd,Nathan Ellis,Overstep,RunOut,360.0,0
20247,15187,1,20,Jos Buttler,Yash Dayal,Overstep,RunOut,813.0,1
20248,15187,1,20,Ravichandran Ashwin,Yash Dayal,wide,RunOut,6423.0,0
23402,15609,1,10,Milton Shumba,Mohammad Nabi,wide,Stump,4876.0,0
23716,15610,1,20,Najibullah Zadran,Tendai Chatara,wide,RunOut,2533.0,0
24021,15611,2,8,Sikandar Raza,Sharafuddin Ashraf,wide,Stump,2984.0,0
35456,16866,1,19,Rizwan CP,Ehsan Khan,wide,Stump,7406.0,0


In [52]:
# 1. Identify legal deliveries

selected_ball["is_legal_ball"] = (
    ~selected_ball["extras_type"].isin(["wide", "Overstep"])
)

# 2. Identify wickets credited to the bowler

bowler_credited_dismissals = [
    "Bowled",
    "Caught",
    "LBW",
    "Stump",
    "Hit Wicket"
]

selected_ball["is_bowler_credited_wicket"] = (
    selected_ball["is_wicket"].eq(True) &
    selected_ball["dismissal_type"].isin(
        bowler_credited_dismissals
    )
)

In [56]:
# Standardize delivery categories
selected_ball["delivery_type"] = (
    selected_ball["extras_type"].fillna("normal")
)

# Classify only categories we can establish
legal_map = {
    "normal": True,
    "byes": True,
    "legbyes": True,
    "wide": False,
    "Overstep": False,
    "other": pd.NA,
    "Penalty": pd.NA
}

selected_ball["is_legal_ball"] = (
    selected_ball["delivery_type"]
    .map(legal_map)
    .astype("boolean")
)

# Audit the classification
print("Legal:", selected_ball["is_legal_ball"].eq(True).sum())
print("Illegal:", selected_ball["is_legal_ball"].eq(False).sum())
print("Unknown:", selected_ball["is_legal_ball"].isna().sum())

display(
    selected_ball.groupby(
        ["delivery_type", "is_legal_ball"],
        dropna=False
    ).size().reset_index(name="delivery_count")
)

Legal: 185714
Illegal: 8039
Unknown: 261


,delivery_type,is_legal_ball,delivery_count
0,Overstep,False,565
1,Penalty,<NA>,1
2,byes,True,505
3,legbyes,True,2661
4,normal,True,182548
5,other,<NA>,260
6,wide,False,7474


In [54]:
# Inspect all records classified as "other" or "Penalty"

unknown_extras = selected_ball[
    selected_ball["extras_type"].isin(["other", "Penalty"])
]

cols_to_inspect = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "innings_ball_num",
    "striker_name",
    "bowler_name",
    "runs",
    "is_wicket",
    "dismissal_type",
    "extras_type",
    "extras1_runs",
    "extras_2_runs"
]

print("Total records:", len(unknown_extras))

print("\nCategory counts:")
display(unknown_extras["extras_type"].value_counts())

print("\nAll Penalty records:")
display(
    unknown_extras[
        unknown_extras["extras_type"] == "Penalty"
    ][cols_to_inspect]
)

print("\nFirst 30 'other' records:")
display(
    unknown_extras[
        unknown_extras["extras_type"] == "other"
    ][cols_to_inspect].head(30)
)

Total records: 261

Category counts:


extras_type
other      260
Penalty      1
Name: count, dtype: int64


All Penalty records:


,match_id,innings,overid,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,is_wicket,dismissal_type,extras_type,extras1_runs,extras_2_runs
34203,16753,2,6,4,34,Harry Tector,Fareed Malik,0,False,NaN,Penalty,7,0



First 30 'other' records:


,match_id,innings,overid,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,is_wicket,dismissal_type,extras_type,extras1_runs,extras_2_runs
532,13513,1,10,1,57,Rovman Powell,Tymal Mills,0,False,NaN,other,5,0
1079,13515,1,16,6,99,Rovman Powell,Saqib Mahmood,0,False,NaN,other,5,0
1226,13515,2,19,5,119,Sam Billings,Sheldon Cottrell,2,False,NaN,other,5,0
2588,13614,1,18,5,111,Dasun Shanaka,Mitchell Starc,0,False,NaN,other,5,4
2609,13614,2,2,1,7,Ashton Agar,Nuwan Thushara,0,False,NaN,other,5,0
3309,14094,1,19,7,119,Mahendra Singh Dhoni,Shivam Mavi,6,False,NaN,other,5,0
4413,14098,2,20,1,119,Bhuvneshwar Kumar,Nathan Coulter Nile,0,False,NaN,other,5,0
5012,14101,1,17,4,105,Kagiso Rabada,Tim Southee,0,False,NaN,other,5,0
5365,14102,2,19,9,122,Kieron Pollard,Prasidh Krishna,0,True,RunOut,other,5,0
6575,14107,2,16,5,95,Patrick Cummins,Daniel Sams,2,False,NaN,other,5,0


In [55]:
# Check whether "other" records occur beyond the normal 6-ball over

sequence_audit = (
    selected_ball
    .assign(
        delivery_type=selected_ball["extras_type"].fillna("normal"),
        beyond_six=selected_ball["over_ball_num"] > 6
    )
    .groupby("delivery_type")
    .agg(
        total_deliveries=("match_id", "size"),
        beyond_six_deliveries=("beyond_six", "sum"),
        max_over_ball_num=("over_ball_num", "max")
    )
    .reset_index()
)

display(sequence_audit)

print("\nOther category: over-ball distribution")
display(
    selected_ball.loc[
        selected_ball["extras_type"] == "other",
        "over_ball_num"
    ].value_counts().sort_index()
)

print("\nPenalty record with match details")
penalty_match_id = unknown_extras.loc[
    unknown_extras["extras_type"] == "Penalty",
    "match_id"
].iloc[0]

display(
    selected_matches.loc[
        selected_matches["match_id"] == penalty_match_id
    ]
)

,delivery_type,total_deliveries,beyond_six_deliveries,max_over_ball_num
0,Overstep,565,22,9
1,Penalty,1,0,4
2,byes,505,33,8
3,legbyes,2661,140,9
4,normal,182548,7505,13
5,other,260,20,9
6,wide,7474,463,12



Other category: over-ball distribution


over_ball_num
1    43
2    34
3    31
4    44
5    47
6    41
7    14
8     5
9     1
Name: count, dtype: int64


Penalty record with match details


,match_id,comp_id,match_number,match_date,stage,venue_id,venue_country,home_team_id,home_team,away_team_id,...,away_team_captain_id,toss_win_team_id,toss_opted,man_of_the_match,first_inn_revised_overs,second_inn_revised_overs,revised_target,knock_out,win_team_id,result
143,16753,1543,4,2022-08-15,League,120,United Kingdom,23,Ireland,78,...,1107.0,23.0,Fielding,4722,66.0,NaN,NaN,0,78.0,Afghanistan Won by 27 Runs


In [57]:

# STEP 1: Prepare the analytical dataset

# 1. Identify dismissals credited to bowlers
bowler_credited_dismissals = [
    "Bowled",
    "Caught",
    "LBW",
    "Stump",
    "Hit Wicket"
]

selected_ball["is_bowler_credited_wicket"] = (
    selected_ball["is_wicket"].eq(True) &
    selected_ball["dismissal_type"].isin(
        bowler_credited_dismissals
    )
)

# 2. Identify regulation innings
selected_ball["is_regulation_innings"] = (
    selected_ball["innings"].isin([1, 2])
)

# 3. Select analytical columns
analysis_columns = [
    "match_id",
    "innings",
    "innings_ball_num",
    "batting_team_id",
    "bowling_team_id",
    "striker_id",
    "striker_name",
    "non_striker_id",
    "non_striker_name",
    "bowler_id",
    "bowler_name",
    "batter_runs",
    "is_wicket",
    "is_bowler_credited_wicket",
    "is_legal_ball",
    "is_regulation_innings",
    "delivery_type",
    "dismissal_type"
]

analysis_ball = selected_ball[analysis_columns].copy()

# 4. Validate
print("ANALYTICAL DATASET CREATED")
print("--------------------------")
print("Total records:", len(analysis_ball))
print("Total columns:", len(analysis_ball.columns))

print(
    "Bowler-credited wickets:",
    analysis_ball["is_bowler_credited_wicket"].sum()
)

print(
    "Regulation innings deliveries:",
    analysis_ball["is_regulation_innings"].sum()
)

print(
    "Super Over deliveries:",
    (~analysis_ball["is_regulation_innings"]).sum()
)

print("\nMissing values:")
display(analysis_ball.isna().sum())

print("\nFirst 5 records:")
display(analysis_ball.head())


ANALYTICAL DATASET CREATED
--------------------------
Total records: 194014
Total columns: 18
Bowler-credited wickets: 9905
Regulation innings deliveries: 193988
Super Over deliveries: 26

Missing values:


match_id                          0
innings                           0
innings_ball_num                  0
batting_team_id                   0
bowling_team_id                   0
striker_id                        0
striker_name                      0
non_striker_id                    0
non_striker_name                  0
bowler_id                         0
bowler_name                       0
batter_runs                       0
is_wicket                         0
is_bowler_credited_wicket         0
is_legal_ball                   261
is_regulation_innings             0
delivery_type                     0
dismissal_type               183528
dtype: int64


First 5 records:


,match_id,innings,innings_ball_num,batting_team_id,bowling_team_id,striker_id,striker_name,non_striker_id,non_striker_name,bowler_id,bowler_name,batter_runs,is_wicket,is_bowler_credited_wicket,is_legal_ball,is_regulation_innings,delivery_type,dismissal_type
0,13511,1,1,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,0,False,False,True,True,normal,NaN
1,13511,1,2,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,0,False,False,True,True,normal,NaN
2,13511,1,3,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,0,False,False,True,True,normal,NaN
3,13511,1,4,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,0,False,False,True,True,normal,NaN
4,13511,1,5,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,6,False,False,True,True,normal,NaN


In [60]:

# ============================================================
# STEP 2: MATCH STATE RECONSTRUCTION
# SupeRR Selector Season 2
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Work only with regulation innings (1 and 2)
# ------------------------------------------------------------

match_state = analysis_ball[
    analysis_ball["is_regulation_innings"] == True
].copy()

# Sort deliveries in chronological match order
match_state = match_state.sort_values(
    ["match_id", "innings", "innings_ball_num"]
).reset_index(drop=True)


# ------------------------------------------------------------
# 2. Create numeric indicators safely
# ------------------------------------------------------------

# Missing wicket values are treated as no confirmed wicket.
match_state["wicket_value"] = (
    match_state["is_wicket"]
    .eq(True)
    .fillna(False)
    .astype(int)
)

# Count only deliveries explicitly classified as legal.
match_state["legal_ball_value"] = (
    match_state["is_legal_ball"]
    .eq(True)
    .fillna(False)
    .astype(int)
)

# Keep track of deliveries with unknown legal-ball status.
match_state["unknown_ball_value"] = (
    match_state["is_legal_ball"]
    .isna()
    .astype(int)
)


# ------------------------------------------------------------
# 3. Group deliveries by match and innings
# ------------------------------------------------------------

innings_groups = match_state.groupby(
    ["match_id", "innings"],
    sort=False
)


# ------------------------------------------------------------
# 4. Calculate match state BEFORE each delivery
# ------------------------------------------------------------

# Batter runs scored before the current delivery.
# NOTE: These are batter runs, NOT total team runs.

match_state["runs_before"] = (
    innings_groups["batter_runs"].cumsum()
    - match_state["batter_runs"]
)

# Wickets lost before the current delivery.

match_state["wickets_before"] = (
    innings_groups["wicket_value"].cumsum()
    - match_state["wicket_value"]
)

# Known legal balls bowled before the current delivery.

match_state["legal_balls_before"] = (
    innings_groups["legal_ball_value"].cumsum()
    - match_state["legal_ball_value"]
)

# Unknown delivery classifications before the current delivery.

match_state["unknown_balls_before"] = (
    innings_groups["unknown_ball_value"].cumsum()
    - match_state["unknown_ball_value"]
)


# ------------------------------------------------------------
# 5. Calculate completed overs
# ------------------------------------------------------------

# This is based on known legal balls only.
# It is provisional if unknown deliveries occurred earlier.

match_state["overs_completed"] = (
    match_state["legal_balls_before"] / 6
)


# ------------------------------------------------------------
# 6. Add match-state confidence indicator
# ------------------------------------------------------------

# State is complete when no earlier delivery has an
# unknown legal-ball classification.

match_state["state_quality"] = (
    match_state["unknown_balls_before"]
    .eq(0)
    .map({
        True: "Complete",
        False: "Uncertain"
    })
)


# ------------------------------------------------------------
# 7. Validation summary
# ------------------------------------------------------------

print("=" * 55)
print("       MATCH STATE RECONSTRUCTION")
print("=" * 55)

print("Regulation deliveries:", len(match_state))

print(
    "Total batter runs:",
    match_state["batter_runs"].sum()
)

print(
    "Total wickets:",
    match_state["wicket_value"].sum()
)

print(
    "Known legal deliveries:",
    match_state["legal_ball_value"].sum()
)

print(
    "Unknown legal-ball classifications:",
    match_state["unknown_ball_value"].sum()
)

print(
    "Deliveries with uncertain prior ball count:",
    (match_state["state_quality"] == "Uncertain").sum()
)


# ------------------------------------------------------------
# 8. Display first 15 reconstructed deliveries
# ------------------------------------------------------------

display(
    match_state[
        [
            "match_id",
            "innings",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "batter_runs",
            "runs_before",
            "wickets_before",
            "legal_balls_before",
            "unknown_balls_before",
            "overs_completed",
            "state_quality"
        ]
    ].head(15)
)


# ------------------------------------------------------------
# 9. Validate first delivery of every innings
# ------------------------------------------------------------

first_deliveries = (
    match_state
    .groupby(["match_id", "innings"], sort=False)
    .head(1)
)

assert (first_deliveries["runs_before"] == 0).all()
assert (first_deliveries["wickets_before"] == 0).all()
assert (first_deliveries["legal_balls_before"] == 0).all()
assert (first_deliveries["unknown_balls_before"] == 0).all()

print("\nFirst-delivery validation: PASSED")

print("\nStep 2 completed successfully.")


       MATCH STATE RECONSTRUCTION
Regulation deliveries: 193988
Total batter runs: 261082
Total wickets: 10478
Known legal deliveries: 185691
Unknown legal-ball classifications: 260
Deliveries with uncertain prior ball count: 9262


,match_id,innings,innings_ball_num,striker_name,bowler_name,batter_runs,runs_before,wickets_before,legal_balls_before,unknown_balls_before,overs_completed,state_quality
0,13511,1,1,Jason Roy,Sheldon Cottrell,0,0,0,0,0,0.000000,Complete
1,13511,1,2,Jason Roy,Sheldon Cottrell,0,0,0,1,0,0.166667,Complete
2,13511,1,3,Jason Roy,Sheldon Cottrell,0,0,0,2,0,0.333333,Complete
3,13511,1,4,Jason Roy,Sheldon Cottrell,0,0,0,3,0,0.500000,Complete
4,13511,1,5,Jason Roy,Sheldon Cottrell,6,0,0,4,0,0.666667,Complete
5,13511,1,6,Jason Roy,Sheldon Cottrell,0,6,0,5,0,0.833333,Complete
6,13511,1,7,Tom Banton,Jason Holder,0,6,1,6,0,1.000000,Complete
7,13511,1,8,Tom Banton,Jason Holder,0,6,1,7,0,1.166667,Complete
8,13511,1,9,Tom Banton,Jason Holder,0,6,1,8,0,1.333333,Complete
9,13511,1,10,Tom Banton,Jason Holder,4,6,1,9,0,1.500000,Complete



First-delivery validation: PASSED

Step 2 completed successfully.


In [61]:

# ============================================================
# STEP 3A: AUDIT INNINGS OVERS
# ============================================================

overs_audit = selected_matches[
    [
        "match_id",
        "first_inn_revised_overs",
        "second_inn_revised_overs"
    ]
].copy()

print("FIRST INNINGS REVISED OVERS")
print(
    overs_audit["first_inn_revised_overs"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nSECOND INNINGS REVISED OVERS")
print(
    overs_audit["second_inn_revised_overs"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nNon-null revised overs:")

print(
    "First innings:",
    overs_audit["first_inn_revised_overs"].notna().sum()
)

print(
    "Second innings:",
    overs_audit["second_inn_revised_overs"].notna().sum()
)


FIRST INNINGS REVISED OVERS
first_inn_revised_overs
30.0       2
42.0       1
48.0       1
54.0       4
60.0       1
66.0       3
72.0       1
78.0       1
84.0       3
90.0       2
102.0      2
108.0      2
114.0      2
NaN      828
Name: count, dtype: int64

SECOND INNINGS REVISED OVERS
second_inn_revised_overs
30.0       2
42.0       2
48.0       3
54.0       2
60.0       2
66.0       1
72.0       1
78.0       1
84.0       1
90.0       5
102.0      2
114.0      3
NaN      828
Name: count, dtype: int64

Non-null revised overs:
First innings: 25
Second innings: 25


In [62]:

# ============================================================
# STEP 3B: PRESSURE DETECTION ENGINE
# ============================================================

import numpy as np
import pandas as pd

# Work on a separate copy
pressure_data = match_state.copy()

# ------------------------------------------------------------
# 1. Define provisional T20 reference
# ------------------------------------------------------------

# 20 overs = 120 legal balls.
# This is a reference assumption, not a verified scheduled
# length for every individual match.

REFERENCE_LEGAL_BALLS = 120

# ------------------------------------------------------------
# 2. Calculate innings progress
# ------------------------------------------------------------

pressure_data["innings_progress"] = (
    pressure_data["legal_balls_before"]
    / REFERENCE_LEGAL_BALLS
)

# Cap progress at 100%.
pressure_data["innings_progress"] = (
    pressure_data["innings_progress"].clip(0, 1)
)

# ------------------------------------------------------------
# 3. Calculate wicket pressure
# ------------------------------------------------------------

pressure_data["wicket_pressure"] = (
    pressure_data["wickets_before"] / 10
).clip(0, 1)

# ------------------------------------------------------------
# 4. Calculate pressure score
# ------------------------------------------------------------

pressure_data["pressure_score"] = (
    100 * (
        0.60 * pressure_data["innings_progress"]
        + 0.40 * pressure_data["wicket_pressure"]
    )
)

# ------------------------------------------------------------
# 5. Assign pressure categories
# ------------------------------------------------------------

pressure_data["pressure_level"] = pd.cut(
    pressure_data["pressure_score"],
    bins=[-0.01, 25, 50, 75, 100],
    labels=[
        "Low",
        "Moderate",
        "High",
        "Extreme"
    ]
)

# ------------------------------------------------------------
# 6. Mark state reliability
# ------------------------------------------------------------

pressure_data["pressure_reliability"] = np.where(
    pressure_data["state_quality"] == "Complete",
    "Reliable ball count",
    "Uncertain ball count"
)

# ------------------------------------------------------------
# 7. Create pressure eligibility flag
# ------------------------------------------------------------

pressure_data["pressure_eligible"] = (
    pressure_data["state_quality"] == "Complete"
)

# ------------------------------------------------------------
# 8. Display summary
# ------------------------------------------------------------

print("=" * 55)
print("          PRESSURE DETECTION SUMMARY")
print("=" * 55)

print("\nTotal deliveries:", len(pressure_data))

print(
    "\nEligible deliveries:",
    pressure_data["pressure_eligible"].sum()
)

print(
    "Uncertain deliveries:",
    (~pressure_data["pressure_eligible"]).sum()
)

print("\nPressure level distribution:")

print(
    pressure_data["pressure_level"]
    .value_counts()
    .sort_index()
)

print("\nPressure score statistics:")

print(
    pressure_data["pressure_score"].describe()
)

print("\nSample pressure records:")

display(
    pressure_data[
        [
            "match_id",
            "innings",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs_before",
            "wickets_before",
            "legal_balls_before",
            "pressure_score",
            "pressure_level",
            "pressure_reliability"
        ]
    ].head(15)
)


          PRESSURE DETECTION SUMMARY

Total deliveries: 193988

Eligible deliveries: 184726
Uncertain deliveries: 9262

Pressure level distribution:
pressure_level
Low         65197
Moderate    61366
High        50875
Extreme     16550
Name: count, dtype: int64

Pressure score statistics:
count    193988.000000
mean         39.280118
std          24.419645
min           0.000000
25%          18.500000
50%          38.000000
75%          59.000000
max          95.500000
Name: pressure_score, dtype: float64

Sample pressure records:


,match_id,innings,innings_ball_num,striker_name,bowler_name,runs_before,wickets_before,legal_balls_before,pressure_score,pressure_level,pressure_reliability
0,13511,1,1,Jason Roy,Sheldon Cottrell,0,0,0,0.0,Low,Reliable ball count
1,13511,1,2,Jason Roy,Sheldon Cottrell,0,0,1,0.5,Low,Reliable ball count
2,13511,1,3,Jason Roy,Sheldon Cottrell,0,0,2,1.0,Low,Reliable ball count
3,13511,1,4,Jason Roy,Sheldon Cottrell,0,0,3,1.5,Low,Reliable ball count
4,13511,1,5,Jason Roy,Sheldon Cottrell,0,0,4,2.0,Low,Reliable ball count
5,13511,1,6,Jason Roy,Sheldon Cottrell,6,0,5,2.5,Low,Reliable ball count
6,13511,1,7,Tom Banton,Jason Holder,6,1,6,7.0,Low,Reliable ball count
7,13511,1,8,Tom Banton,Jason Holder,6,1,7,7.5,Low,Reliable ball count
8,13511,1,9,Tom Banton,Jason Holder,6,1,8,8.0,Low,Reliable ball count
9,13511,1,10,Tom Banton,Jason Holder,6,1,9,8.5,Low,Reliable ball count


In [63]:

# ============================================================
# STEP 4A: FILTER HIGH-PRESSURE DELIVERIES
# ============================================================

# Keep only reliable reconstructed states
# and High / Extreme pressure situations.

high_pressure_data = pressure_data[
    (pressure_data["pressure_eligible"] == True) &
    (pressure_data["pressure_score"] >= 50)
].copy()

# ------------------------------------------------------------
# 1. Basic delivery statistics
# ------------------------------------------------------------

print("=" * 55)
print("       HIGH-PRESSURE DELIVERY DATASET")
print("=" * 55)

print("Total high-pressure deliveries:", len(high_pressure_data))

print(
    "Unique matches:",
    high_pressure_data["match_id"].nunique()
)

print(
    "Unique batters:",
    high_pressure_data["striker_id"].nunique()
)

print(
    "Unique bowlers:",
    high_pressure_data["bowler_id"].nunique()
)

# ------------------------------------------------------------
# 2. Innings distribution
# ------------------------------------------------------------

print("\nHigh-pressure deliveries by innings:")

print(
    high_pressure_data["innings"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 3. Pressure-level distribution
# ------------------------------------------------------------

print("\nPressure-level distribution:")

print(
    high_pressure_data["pressure_level"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 4. Check required player identifiers
# ------------------------------------------------------------

print("\nMissing player identifiers:")

print(
    "Missing striker IDs:",
    high_pressure_data["striker_id"].isna().sum()
)

print(
    "Missing bowler IDs:",
    high_pressure_data["bowler_id"].isna().sum()
)

# ------------------------------------------------------------
# 5. Check available bowler-credited wicket data
# ------------------------------------------------------------

print("\nBowler-credited wickets in high pressure:")

print(
    high_pressure_data["is_bowler_credited_wicket"].sum()
)

print("\nHigh-pressure dataset is ready for player analysis.")


       HIGH-PRESSURE DELIVERY DATASET
Total high-pressure deliveries: 62219
Unique matches: 816
Unique batters: 729
Unique bowlers: 536

High-pressure deliveries by innings:
innings
1    34292
2    27927
Name: count, dtype: int64

Pressure-level distribution:
pressure_level
Low             0
Moderate      719
High        46821
Extreme     14679
Name: count, dtype: int64

Missing player identifiers:
Missing striker IDs: 0
Missing bowler IDs: 0

Bowler-credited wickets in high pressure:
4072

High-pressure dataset is ready for player analysis.


In [65]:

# ============================================================
# STEP 4B: BATTER PERFORMANCE ANALYSIS
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Select eligible high-pressure deliveries
# ------------------------------------------------------------

batter_data = pressure_data[
    (pressure_data["pressure_eligible"] == True) &
    (pressure_data["pressure_score"] >= 50)
].copy()

# ------------------------------------------------------------
# 2. Identify batting events
# ------------------------------------------------------------

# A wide does not count as a batter ball faced.
# Other delivery types are retained unless classified as wide.

batter_data["batter_ball_faced"] = (
    batter_data["delivery_type"] != "wide"
).astype(int)

# Identify boundaries using batter runs, not source flags.

batter_data["fours"] = (
    batter_data["batter_runs"] == 4
).astype(int)

batter_data["sixes"] = (
    batter_data["batter_runs"] == 6
).astype(int)

# ------------------------------------------------------------
# 3. Aggregate batter statistics
# ------------------------------------------------------------

batter_stats = (
    batter_data
    .groupby(["striker_id", "striker_name"], as_index=False)
    .agg(
        high_pressure_runs=("batter_runs", "sum"),
        balls_faced=("batter_ball_faced", "sum"),
        fours=("fours", "sum"),
        sixes=("sixes", "sum"),
        pressure_deliveries=("match_id", "size"),
        matches_played=("match_id", "nunique")
    )
)

# ------------------------------------------------------------
# 4. Calculate batting performance metrics
# ------------------------------------------------------------

batter_stats["strike_rate"] = np.where(
    batter_stats["balls_faced"] > 0,
    (
        batter_stats["high_pressure_runs"]
        / batter_stats["balls_faced"]
    ) * 100,
    0
)

batter_stats["boundary_runs"] = (
    batter_stats["fours"] * 4
    + batter_stats["sixes"] * 6
)

batter_stats["boundary_percentage"] = np.where(
    batter_stats["high_pressure_runs"] > 0,
    (
        batter_stats["boundary_runs"]
        / batter_stats["high_pressure_runs"]
    ) * 100,
    0
)

# ------------------------------------------------------------
# 5. Apply minimum sample-size requirement
# ------------------------------------------------------------

MIN_BALLS_FACED = 20

batter_shortlist = batter_stats[
    batter_stats["balls_faced"] >= MIN_BALLS_FACED
].copy()

# ------------------------------------------------------------
# 6. Sort by high-pressure runs initially
# ------------------------------------------------------------

batter_shortlist = batter_shortlist.sort_values(
    ["high_pressure_runs", "strike_rate"],
    ascending=[False, False]
).reset_index(drop=True)

batter_shortlist.insert(
    0,
    "provisional_rank",
    range(1, len(batter_shortlist) + 1)
)

# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("=" * 60)
print("           BATTER PERFORMANCE ANALYSIS")
print("=" * 60)

print("Total batters:", len(batter_stats))
print("Batters qualifying for shortlist:", len(batter_shortlist))

print("\nTop 20 batters by high-pressure runs:")

display(
    batter_shortlist[
        [
            "provisional_rank",
            "striker_id",
            "striker_name",
            "high_pressure_runs",
            "balls_faced",
            "strike_rate",
            "fours",
            "sixes",
            "boundary_percentage",
            "matches_played"
        ]
    ].head(20).round(2)
)

# ------------------------------------------------------------
# 8. Save intermediate batter statistics
# ------------------------------------------------------------
from pathlib import Path

# Create output directory automatically if it doesn't exist
OUTPUT_DIR = Path("outputs") / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


           BATTER PERFORMANCE ANALYSIS
Total batters: 729
Batters qualifying for shortlist: 436

Top 20 batters by high-pressure runs:


,provisional_rank,striker_id,striker_name,high_pressure_runs,balls_faced,strike_rate,fours,sixes,boundary_percentage,matches_played
0,1,7897,Namboori Tilak Varma,1478,895,165.14,107,82,62.25,76
1,2,6396,Tim David,1434,807,177.70,100,103,70.99,76
2,3,4708,Rinku Singh,1334,818,163.08,109,71,64.62,65
3,4,4755,Shivam Dube,1319,787,167.60,86,91,67.48,66
4,5,3330,Hardik Pandya,1307,883,148.02,103,64,60.90,71
5,6,4986,Rovman Powell,1234,779,158.41,72,91,67.59,64
6,7,2876,Axar Patel,1149,791,145.26,82,57,58.31,79
7,8,3125,Nicholas Pooran,1143,648,176.39,79,86,72.79,55
8,9,3297,Shimron Hetmyer,1122,744,150.81,75,69,63.64,68
9,10,1699,Suryakumar Yadav,1116,575,194.09,102,73,75.81,48


In [66]:

# ============================================================
# STEP 4C: FINAL BATTER RANKING ENGINE
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Copy the qualifying batter shortlist
# ------------------------------------------------------------

final_batters = batter_shortlist.copy()

# ------------------------------------------------------------
# 2. Calculate percentile scores
# ------------------------------------------------------------

# Each metric is converted to a percentile between 0 and 1.
# Higher values indicate better performance.

final_batters["runs_percentile"] = (
    final_batters["high_pressure_runs"].rank(pct=True)
)

final_batters["strike_rate_percentile"] = (
    final_batters["strike_rate"].rank(pct=True)
)

final_batters["boundary_percentile"] = (
    final_batters["boundary_percentage"].rank(pct=True)
)

# ------------------------------------------------------------
# 3. Calculate sample reliability
# ------------------------------------------------------------

# Full sample reliability is reached at 100 balls.

final_batters["sample_reliability"] = (
    final_batters["balls_faced"] / 100
).clip(upper=1)

# ------------------------------------------------------------
# 4. Calculate composite batter score
# ------------------------------------------------------------

final_batters["batter_score"] = (
    0.40 * final_batters["runs_percentile"]
    + 0.30 * final_batters["strike_rate_percentile"]
    + 0.15 * final_batters["boundary_percentile"]
    + 0.15 * final_batters["sample_reliability"]
) * 100

# ------------------------------------------------------------
# 5. Generate final ranking
# ------------------------------------------------------------

final_batters = final_batters.sort_values(
    ["batter_score", "high_pressure_runs"],
    ascending=[False, False]
).reset_index(drop=True)

final_batters["final_rank"] = (
    final_batters.index + 1
)

# ------------------------------------------------------------
# 6. Extract Top 10 batters
# ------------------------------------------------------------

top_10_batters = final_batters.head(10).copy()

# ------------------------------------------------------------
# 7. Display final ranking
# ------------------------------------------------------------

print("=" * 65)
print("             TOP 10 BATTERS UNDER PRESSURE")
print("=" * 65)

display(
    top_10_batters[
        [
            "final_rank",
            "striker_id",
            "striker_name",
            "batter_score",
            "high_pressure_runs",
            "balls_faced",
            "strike_rate",
            "fours",
            "sixes",
            "boundary_percentage",
            "matches_played"
        ]
    ].round(2)
)

# ------------------------------------------------------------
# 8. Save final results
# ------------------------------------------------------------

OUTPUT_DIR = Path("outputs") / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_batters.to_csv(
    OUTPUT_DIR / "final_batter_rankings.csv",
    index=False
)

top_10_batters.to_csv(
    OUTPUT_DIR / "top_10_batters.csv",
    index=False
)

print("\nFinal batter rankings saved.")
print("Location:", OUTPUT_DIR.resolve())


             TOP 10 BATTERS UNDER PRESSURE


,final_rank,striker_id,striker_name,batter_score,high_pressure_runs,balls_faced,strike_rate,fours,sixes,boundary_percentage,matches_played
0,1,1699,Suryakumar Yadav,97.80,1116,575,194.09,102,73,75.81,48
1,2,6396,Tim David,95.71,1434,807,177.70,100,103,70.99,76
2,3,3125,Nicholas Pooran,95.37,1143,648,176.39,79,86,72.79,55
3,4,1318,Andre Russell,94.70,1035,600,172.50,65,94,79.61,53
4,5,3647,Shreyas Iyer,93.80,880,469,187.63,53,63,67.05,41
5,6,4260,Heinrich Klaasen,93.11,1064,579,183.77,78,64,65.41,50
6,7,813,Jos Buttler,92.92,697,377,184.88,64,36,67.72,35
7,8,8101,Ashutosh Sharma,91.89,522,293,178.16,36,38,71.26,24
8,9,2639,Marcus Stoinis,91.87,929,549,169.22,67,63,69.54,59
9,10,6280,Harry Brook,91.79,554,294,188.44,51,28,67.15,24



Final batter rankings saved.
Location: C:\Users\Dell\Downloads\SupeRR_Selector_Season2\notebooks\outputs\tables


In [68]:

# ============================================================
# STEP 5A: BOWLER PERFORMANCE ANALYSIS (CORRECTED)
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# 1. Select eligible high-pressure deliveries
bowler_data = pressure_data[
    (pressure_data["pressure_eligible"] == True) &
    (pressure_data["pressure_score"] >= 50)
].copy()

# 2. Count only known legal balls
bowler_data["known_legal_balls"] = (
    bowler_data["is_legal_ball"]
    .eq(True)
    .fillna(False)
    .astype(int)
)

# 3. Count only bowler-credited wickets
bowler_data["bowler_wicket"] = (
    bowler_data["is_bowler_credited_wicket"]
    .eq(True)
    .fillna(False)
    .astype(int)
)

# 4. Aggregate bowler statistics
bowler_stats = (
    bowler_data
    .groupby(["bowler_id", "bowler_name"], as_index=False)
    .agg(
        high_pressure_wickets=("bowler_wicket", "sum"),
        legal_balls=("known_legal_balls", "sum"),
        pressure_deliveries=("match_id", "size"),
        matches_played=("match_id", "nunique")
    )
)

# 5. Calculate wickets per 24 known legal balls
bowler_stats["wickets_per_24_balls"] = np.where(
    bowler_stats["legal_balls"] > 0,
    (
        bowler_stats["high_pressure_wickets"]
        / bowler_stats["legal_balls"]
    ) * 24,
    0
)

# 6. Apply minimum sample requirement
MIN_LEGAL_BALLS = 24

bowler_shortlist = bowler_stats[
    bowler_stats["legal_balls"] >= MIN_LEGAL_BALLS
].copy()

# 7. Sort provisional rankings
bowler_shortlist = bowler_shortlist.sort_values(
    ["high_pressure_wickets", "wickets_per_24_balls"],
    ascending=[False, False]
).reset_index(drop=True)

bowler_shortlist.insert(
    0, "provisional_rank",
    range(1, len(bowler_shortlist) + 1)
)

# 8. Display results
print("=" * 60)
print("BOWLER PERFORMANCE ANALYSIS")
print("=" * 60)

print("Total bowlers:", len(bowler_stats))
print("Qualifying bowlers:", len(bowler_shortlist))

display(
    bowler_shortlist[
        [
            "provisional_rank",
            "bowler_id",
            "bowler_name",
            "high_pressure_wickets",
            "legal_balls",
            "wickets_per_24_balls",
            "pressure_deliveries",
            "matches_played"
        ]
    ].head(20).round(2)
)

# 9. Save results
OUTPUT_DIR = Path("outputs") / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bowler_stats.to_csv(
    OUTPUT_DIR / "bowler_high_pressure_stats.csv",
    index=False
)

bowler_shortlist.to_csv(
    OUTPUT_DIR / "bowler_high_pressure_shortlist.csv",
    index=False
)

print("\nBowler statistics saved successfully.")
print("Location:", OUTPUT_DIR.resolve())


BOWLER PERFORMANCE ANALYSIS
Total bowlers: 536
Qualifying bowlers: 350


,provisional_rank,bowler_id,bowler_name,high_pressure_wickets,legal_balls,wickets_per_24_balls,pressure_deliveries,matches_played
0,1,6464,Arshdeep Singh,93,1260,1.77,1343,126
1,2,4722,Rashid Khan,80,1008,1.90,1024,104
2,3,1612,Yuzvendra Chahal,60,706,2.04,741,80
3,4,6256,Varun Chakravarthy,56,694,1.94,715,76
4,5,2261,Harshal Patel,56,736,1.83,792,64
5,6,3644,Kuldeep Yadav,55,621,2.13,642,68
6,7,8661,Matheesha Pathirana,55,666,1.98,742,47
7,8,7503,Ravi Bishnoi,55,766,1.72,794,84
8,9,2633,Jason Holder,54,751,1.73,799,74
9,10,4900,Wanindu Hasaranga,53,604,2.11,609,66



Bowler statistics saved successfully.
Location: C:\Users\Dell\Downloads\SupeRR_Selector_Season2\notebooks\outputs\tables


In [69]:

# ============================================================
# STEP 5B: FINAL BOWLER RANKING ENGINE
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

final_bowlers = bowler_shortlist.copy()

# 1. Calculate percentile scores
final_bowlers["wickets_percentile"] = (
    final_bowlers["high_pressure_wickets"].rank(pct=True)
)

final_bowlers["wicket_rate_percentile"] = (
    final_bowlers["wickets_per_24_balls"].rank(pct=True)
)

# 2. Calculate sample sufficiency
final_bowlers["sample_sufficiency"] = (
    final_bowlers["legal_balls"] / 120
).clip(upper=1)

# 3. Calculate weighted composite score
final_bowlers["bowler_score"] = (
    0.45 * final_bowlers["wickets_percentile"]
    + 0.30 * final_bowlers["wicket_rate_percentile"]
    + 0.25 * final_bowlers["sample_sufficiency"]
) * 100

# 4. Sort by final score
final_bowlers = final_bowlers.sort_values(
    ["bowler_score", "high_pressure_wickets"],
    ascending=[False, False]
).reset_index(drop=True)

# 5. Assign final ranks
final_bowlers["final_rank"] = (
    final_bowlers.index + 1
)

top_10_bowlers = final_bowlers.head(10).copy()

# 6. Display Top 10
print("=" * 65)
print("             TOP 10 BOWLERS UNDER PRESSURE")
print("=" * 65)

display(
    top_10_bowlers[
        [
            "final_rank",
            "bowler_id",
            "bowler_name",
            "bowler_score",
            "high_pressure_wickets",
            "legal_balls",
            "wickets_per_24_balls",
            "matches_played"
        ]
    ].round(2)
)

# 7. Save final rankings
OUTPUT_DIR = Path("outputs") / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_bowlers.to_csv(
    OUTPUT_DIR / "final_bowler_rankings.csv",
    index=False
)

top_10_bowlers.to_csv(
    OUTPUT_DIR / "top_10_bowlers.csv",
    index=False
)

print("\nFinal bowler rankings saved successfully.")
print("Location:", OUTPUT_DIR.resolve())


             TOP 10 BOWLERS UNDER PRESSURE


,final_rank,bowler_id,bowler_name,bowler_score,high_pressure_wickets,legal_balls,wickets_per_24_balls,matches_played
0,1,829,Mitchell Starc,94.09,28,231,2.91,29
1,2,7364,Haris Rauf,93.89,40,402,2.39,38
2,3,3644,Kuldeep Yadav,93.06,55,621,2.13,68
3,4,1612,Yuzvendra Chahal,92.63,60,706,2.04,80
4,5,4900,Wanindu Hasaranga,92.50,53,604,2.11,66
5,6,6355,Shaheen Afridi,92.48,34,358,2.28,36
6,7,5379,Mark Adair,91.73,31,328,2.27,34
7,8,830,Josh Hazlewood,91.51,41,475,2.07,52
8,9,1318,Andre Russell,90.87,31,343,2.17,39
9,10,8661,Matheesha Pathirana,90.83,55,666,1.98,47



Final bowler rankings saved successfully.
Location: C:\Users\Dell\Downloads\SupeRR_Selector_Season2\notebooks\outputs\tables


In [70]:

# ============================================================
# STEP 6A: FINAL VALIDATION AND PROJECT EXPORT
# ============================================================

import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("outputs") / "tables"
REPORT_DIR = Path("outputs") / "reports"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Validate both Top 10 leaderboards
# ------------------------------------------------------------

assert len(top_10_batters) == 10, "Batter leaderboard must have 10 players."
assert len(top_10_bowlers) == 10, "Bowler leaderboard must have 10 players."

assert top_10_batters["striker_id"].nunique() == 10
assert top_10_bowlers["bowler_id"].nunique() == 10

assert top_10_batters["final_rank"].tolist() == list(range(1, 11))
assert top_10_bowlers["final_rank"].tolist() == list(range(1, 11))

assert top_10_batters["batter_score"].between(0, 100).all()
assert top_10_bowlers["bowler_score"].between(0, 100).all()

print("VALIDATION PASSED")
print("Top 10 batters: 10 unique players")
print("Top 10 bowlers: 10 unique players")

# ------------------------------------------------------------
# 2. Prepare a common leaderboard format
# ------------------------------------------------------------

batters_export = pd.DataFrame({
    "role": "Batter",
    "rank": top_10_batters["final_rank"],
    "player_id": top_10_batters["striker_id"],
    "player_name": top_10_batters["striker_name"],
    "composite_score": top_10_batters["batter_score"],
    "high_pressure_runs": top_10_batters["high_pressure_runs"],
    "high_pressure_wickets": None,
    "balls_or_legal_balls": top_10_batters["balls_faced"],
    "performance_rate": top_10_batters["strike_rate"],
    "matches_played": top_10_batters["matches_played"]
})

bowlers_export = pd.DataFrame({
    "role": "Bowler",
    "rank": top_10_bowlers["final_rank"],
    "player_id": top_10_bowlers["bowler_id"],
    "player_name": top_10_bowlers["bowler_name"],
    "composite_score": top_10_bowlers["bowler_score"],
    "high_pressure_runs": None,
    "high_pressure_wickets": top_10_bowlers["high_pressure_wickets"],
    "balls_or_legal_balls": top_10_bowlers["legal_balls"],
    "performance_rate": top_10_bowlers["wickets_per_24_balls"],
    "matches_played": top_10_bowlers["matches_played"]
})

combined_leaderboard = pd.concat(
    [batters_export, bowlers_export],
    ignore_index=True
)

# ------------------------------------------------------------
# 3. Save final leaderboard files
# ------------------------------------------------------------

top_10_batters.to_csv(
    OUTPUT_DIR / "top_10_batters.csv",
    index=False
)

top_10_bowlers.to_csv(
    OUTPUT_DIR / "top_10_bowlers.csv",
    index=False
)

combined_leaderboard.to_csv(
    OUTPUT_DIR / "combined_top_10_leaderboards.csv",
    index=False
)

# ------------------------------------------------------------
# 4. Create methodology and limitations report
# ------------------------------------------------------------

methodology = """
# SupeRR Selector Season 2
## Methodology and Limitations

### 1. Dataset selection
- Competitions: IPL and T20I.
- Period: 2022 onwards, through the available 2026 data.
- Selected matches: 853.
- Matches with ball-by-ball data: 837.
- Matches without ball-by-ball data: 16.
- Super Over deliveries are excluded from regulation-innings rankings.

### 2. Pressure identification
A proxy pressure score is calculated using:

Pressure Score =
100 * (0.60 * innings progress + 0.40 * wicket pressure)

- Innings progress uses known legal balls before the delivery,
  divided by a reference of 120 balls.
- Wicket pressure uses wickets lost before the delivery,
  divided by 10.
- Both components are capped at 1.
- Deliveries with uncertain prior legal-ball counts are excluded.
- High pressure is defined as a score of 50 or higher.

This is a simplified pressure proxy. It is not a validated
win-probability model and does not include target, required
run rate or actual team score.

### 3. Batter ranking
- Minimum sample: 20 balls faced.
- High-pressure runs: 40%.
- Strike-rate percentile: 30%.
- Boundary-percentage percentile: 15%.
- Sample-sufficiency indicator: 15%.
- Sample sufficiency is capped at 100 balls.

Strike rate = high-pressure batter runs / balls faced * 100.

Boundary percentage = boundary runs / high-pressure batter runs * 100.

Wides are excluded from balls faced. Other and Penalty delivery
classifications are included in the current balls-faced proxy.

### 4. Bowler ranking
- Minimum sample: 24 known legal balls.
- High-pressure bowler-credited wickets percentile: 45%.
- Wickets per 24 known legal balls percentile: 30%.
- Sample-sufficiency indicator: 25%.
- Sample sufficiency is capped at 120 known legal balls.

Only Bowled, Caught, LBW, Stump and Hit Wicket dismissals
are counted as bowler-credited wickets.

### 5. Data limitations
- The source runs field was validated against two scorecards
  and appears to represent batter runs, not total team runs.
- Actual extras-run amounts are not reliably available.
- Economy rate, bowling average and true runs conceded
  are therefore not calculated.
- The 120-ball reference is a simplifying assumption.
- Revised-overs metadata semantics have not been independently
  validated.
- There are 260 deliveries with unknown legal-ball status.
  Later delivery states affected by these unknown records
  are excluded from ranking eligibility.
- The pressure score does not measure actual match-winning
  probability or required run rate.
- Ranking weights and sample thresholds are provisional
  design choices, not statistically trained parameters.

### 6. Interpretation
The composite scores are comparative project indices
under the stated assumptions. They should not be presented
as official player ratings or definitive measures of ability.
"""

report_path = REPORT_DIR / "methodology_and_limitations.md"
report_path.write_text(methodology.strip(), encoding="utf-8")

# ------------------------------------------------------------
# 5. Final output summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PROJECT EXPORT COMPLETED")
print("=" * 60)

print("Batter leaderboard:", OUTPUT_DIR / "top_10_batters.csv")
print("Bowler leaderboard:", OUTPUT_DIR / "top_10_bowlers.csv")
print("Combined leaderboard:", OUTPUT_DIR / "combined_top_10_leaderboards.csv")
print("Methodology report:", report_path)

print("\nTotal ranked players:", len(combined_leaderboard))
print("Batter entries:", (combined_leaderboard["role"] == "Batter").sum())
print("Bowler entries:", (combined_leaderboard["role"] == "Bowler").sum())

display(combined_leaderboard.round(2))


VALIDATION PASSED
Top 10 batters: 10 unique players
Top 10 bowlers: 10 unique players

PROJECT EXPORT COMPLETED
Batter leaderboard: outputs\tables\top_10_batters.csv
Bowler leaderboard: outputs\tables\top_10_bowlers.csv
Combined leaderboard: outputs\tables\combined_top_10_leaderboards.csv
Methodology report: outputs\reports\methodology_and_limitations.md

Total ranked players: 20
Batter entries: 10
Bowler entries: 10


,role,rank,player_id,player_name,composite_score,high_pressure_runs,high_pressure_wickets,balls_or_legal_balls,performance_rate,matches_played
0,Batter,1,1699,Suryakumar Yadav,97.80,1116,None,575,194.09,48
1,Batter,2,6396,Tim David,95.71,1434,None,807,177.70,76
2,Batter,3,3125,Nicholas Pooran,95.37,1143,None,648,176.39,55
3,Batter,4,1318,Andre Russell,94.70,1035,None,600,172.50,53
4,Batter,5,3647,Shreyas Iyer,93.80,880,None,469,187.63,41
5,Batter,6,4260,Heinrich Klaasen,93.11,1064,None,579,183.77,50
6,Batter,7,813,Jos Buttler,92.92,697,None,377,184.88,35
7,Batter,8,8101,Ashutosh Sharma,91.89,522,None,293,178.16,24
8,Batter,9,2639,Marcus Stoinis,91.87,929,None,549,169.22,59
9,Batter,10,6280,Harry Brook,91.79,554,None,294,188.44,24
